[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/calculus/10_multivariable_functions_partials/exercises.ipynb)

# Module 10 — Multivariable Functions and Partial Derivatives — Exercises

Forty fully solved problems in four tiers. Every problem carries a **Statement**, an
**Intuition**, a numbered **Solution**, a boxed answer, a one-line **Key takeaway**, and —
wherever the answer is numeric or algorithmic — a code cell that recomputes it and asserts
the value. Nothing here is quoted from memory.

| Tier | Count | Focus |
| :--- | ---: | :--- |
| `L0 — Concept Checks` | 8 | one-step definitional checks |
| `L1 — Foundations` | 10 | limits, partials, Jacobians, differentiability from the definition |
| `L2 — Applications (AI/ML and Physics)` | 12 | PDEs, thermodynamics, error propagation, autodiff, backpropagation |
| `L3 — Challenge Proofs` | 10 | counterexamples and named theorems |

Run this cell first. It is the only setup the notebook needs; every later code cell
reuses `rng`, the SymPy symbols, and the print options fixed here.

In [1]:
import itertools

import numpy as np
import sympy as sp

np.set_printoptions(precision=6, suppress=True)
rng = np.random.default_rng(0)
x, y, z = sp.symbols("x y z", real=True)
print("numpy", np.__version__, "| sympy", sp.__version__)

numpy 2.4.6 | sympy 1.14.0


The printed versions pin the environment the boxed numbers below were produced in.

## L0 — Concept Checks

### Problem L0.1 — Level Curves of a Rational Surface
**Source:** Adapted from Stewart, *Multivariable Calculus*, 8th Edition, Ch. 14.

**Statement**
Describe and sketch the geometry of the level curves $S_c = \{(x,y) \in \mathbb{R}^2 \mid f(x,y) = c\}$ for the function:

$$
f(x,y) = \frac{x}{y^2 + 1}
$$

for target values $c = -1, 0, 1, 2$.

**Intuition**

Level curves are 2D trajectories along which the surface height $z = f(x,y)$ remains constant at level $c$. Setting $f(x,y) = c$ converts a 3D surface problem into an algebraic equation relating $x$ and $y$.

**Solution**

1. **General Level Curve Formula**:

$$
   \frac{x}{y^2 + 1} = c \iff x = c(y^2 + 1)
$$

2. **Case $c = 0$**:

$$
   x = 0(y^2 + 1) \implies x = 0
$$

   The level curve $S_0$ is the entire $y$-axis.

3. **Case $c = 1$**:

$$
   x = y^2 + 1
$$

   A rightward-opening parabola with vertex at $(1, 0)$ on the $x$-axis.

4. **Case $c = 2$**:

$$
   x = 2(y^2 + 1) = 2y^2 + 2
$$

   A narrower rightward-opening parabola with vertex at $(2, 0)$.

5. **Case $c = -1$**:

$$
   x = -(y^2 + 1) = -y^2 - 1
$$

   A leftward-opening parabola with vertex at $(-1, 0)$.

$$
\boxed{x = c(y^2 + 1); \text{ Parabolas opening along the } x\text{-axis (left for } c\lt 0\text{, right for } c\gt 0\text{, } y\text{-axis for } c=0).}
$$

**Key takeaway** — Level curves of $f(x,y) = \frac{x}{y^2+1}$ form a family of horizontal parabolas opening rightward for $c \gt 0$, leftward for $c \lt 0$, and degenerating into a straight line ($x=0$) for $c = 0$.

Recomputing Problem L0.1 from scratch, to check the boxed answer.

In [2]:
# Every point of the claimed curve x = c(y^2+1) must have f = c exactly.
f = lambda X, Y: X / (Y**2 + 1.0)
y_grid = np.linspace(-2.0, 2.0, 9)
for c in (-1.0, 0.0, 1.0, 2.0):
    X = c * (y_grid**2 + 1.0)
    resid = np.max(np.abs(f(X, y_grid) - c))
    print(f"c = {c:5.1f}   vertex x = {c:5.1f}   max |f(c(y^2+1), y) - c| = {resid:.2e}")
    assert resid < 1e-15

c =  -1.0   vertex x =  -1.0   max |f(c(y^2+1), y) - c| = 0.00e+00
c =   0.0   vertex x =   0.0   max |f(c(y^2+1), y) - c| = 0.00e+00
c =   1.0   vertex x =   1.0   max |f(c(y^2+1), y) - c| = 0.00e+00
c =   2.0   vertex x =   2.0   max |f(c(y^2+1), y) - c| = 0.00e+00


The residual is exactly zero in floating point: `c(y^2+1)/(y^2+1)` cancels without rounding, so the parametrisation of the level curve is exact, not approximate.

### Problem L0.2 — A Limit That Depends on the Slope
**Source:** Adapted from Apostol, *Mathematical Analysis*, 2nd Edition, Ch. 4.

**Statement**
Analyze the limit as $(x,y) \to (0,0)$ along straight lines $y = m x$ for the function:

$$
f(x,y) = \frac{x y}{x^2 + y^2}, \quad (x,y) \neq (0,0)
$$

Conclude whether $\lim_{(x,y) \to (0,0)} f(x,y)$ exists.

**Intuition**

A multivariable limit exists if and only if $f(x,y)$ approaches the exact same value $L$ regardless of the path chosen to approach $(0,0)$. Testing straight-line paths $y = m x$ parameterized by slope $m$ tests an entire family of linear approaches.

**Solution**

1. **Substitute $y = m x$ into $f(x,y)$ for $x \neq 0$**:

$$
   f(x, mx) = \frac{x (m x)}{x^2 + (m x)^2} = \frac{m x^2}{x^2 (1 + m^2)} = \frac{m}{1 + m^2}
$$

2. **Evaluate the Limit along Path $y = m x$**:

$$
   \lim_{x \to 0} f(x, mx) = \frac{m}{1 + m^2}
$$

3. **Path-Dependence Analysis**:
   - For $m = 0$ ($x$-axis), the limit is $\frac{0}{1} = 0$.
   - For $m = 1$ (line $y = x$), the limit is $\frac{1}{1 + 1^2} = \frac{1}{2}$.
   - For $m = -1$ (line $y = -x$), the limit is $\frac{-1}{1 + 1} = -\frac{1}{2}$.

4. **Conclusion**:
   Since different paths yield different limit values ($0 \neq \frac{1}{2} \neq -\frac{1}{2}$), the multivariable limit **does not exist**.

$$
\boxed{\text{Limit along } y=mx \text{ is } \frac{m}{1+m^2}; \text{ Limit does not exist at } (0,0).}
$$

**Key takeaway** — If a limit along $y = mx$ depends on the parameter $m$, the multivariable limit fails to exist due to path dependence.

Recomputing Problem L0.2 from scratch, to check the boxed answer.

In [3]:
# Along y = m x the quotient should be the constant m/(1+m^2), independent of x.
f = lambda X, Y: X * Y / (X**2 + Y**2)
xs = np.array([1e-2, 1e-4, 1e-6, 1e-8])
for m in (-1.0, 0.0, 0.5, 1.0, 3.0):
    vals = f(xs, m * xs)
    print(f"m = {m:5.2f}   f(x, mx) = {vals}   m/(1+m^2) = {m/(1+m**2):+.6f}")
    assert np.allclose(vals, m / (1 + m**2), atol=1e-15)
print("distinct slope limits:", sorted({round(m / (1 + m**2), 6) for m in (-1.0, 0.0, 1.0)}))

m = -1.00   f(x, mx) = [-0.5 -0.5 -0.5 -0.5]   m/(1+m^2) = -0.500000
m =  0.00   f(x, mx) = [0. 0. 0. 0.]   m/(1+m^2) = +0.000000
m =  0.50   f(x, mx) = [0.4 0.4 0.4 0.4]   m/(1+m^2) = +0.400000
m =  1.00   f(x, mx) = [0.5 0.5 0.5 0.5]   m/(1+m^2) = +0.500000
m =  3.00   f(x, mx) = [0.3 0.3 0.3 0.3]   m/(1+m^2) = +0.300000
distinct slope limits: [-0.5, 0.0, 0.5]


The quotient is constant in `x` along each line and takes three different values for `m = -1, 0, 1`. One family of paths already settles the question.

### Problem L0.3 — Partials of a Square-Root Product at the Origin
**Source:** Adapted from Spivak, *Calculus on Manifolds*, Ch. 2.

**Statement**
Consider $f(x,y) = \sqrt{\lvert x y \rvert}$. Compute the partial derivatives $f_x(0,0)$ and $f_y(0,0)$ directly using the limit definition.

**Intuition**

Partial derivatives at a specific point $(a,b)$ are defined by freezing all variables except one and taking a 1D difference quotient limit along the coordinate axis.

**Solution**

1. **Compute $f_x(0,0)$ via Definition**:

$$
   f_x(0,0) = \lim_{h \to 0} \frac{f(0+h, 0) - f(0,0)}{h}
$$

   Evaluate $f(h, 0)$:

$$
   f(h, 0) = \sqrt{\lvert h \cdot 0 \rvert} = \sqrt{0} = 0
$$

   Substitute back:

$$
   f_x(0,0) = \lim_{h \to 0} \frac{0 - 0}{h} = \lim_{h \to 0} 0 = 0
$$

2. **Compute $f_y(0,0)$ via Definition**:

$$
   f_y(0,0) = \lim_{k \to 0} \frac{f(0, 0+k) - f(0,0)}{k} = \lim_{k \to 0} \frac{\sqrt{\lvert 0 \cdot k \rvert} - 0}{k} = \lim_{k \to 0} 0 = 0
$$

$$
\boxed{f_x(0,0) = 0, \quad f_y(0,0) = 0.}
$$

**Key takeaway** — Along the coordinate axes $x=0$ and $y=0$, the function value $f(x,y) = \sqrt{\lvert xy \rvert}$ is identically zero. Thus, the difference quotients are identically zero, yielding $f_x(0,0) = f_y(0,0) = 0$.

Recomputing Problem L0.3 from scratch, to check the boxed answer.

In [4]:
# f vanishes identically on both axes, so both difference quotients are exactly 0.
f = lambda X, Y: np.sqrt(np.abs(X * Y))
for h in (1e-1, 1e-3, 1e-5, 1e-7):
    qx = (f(h, 0.0) - 0.0) / h
    qy = (f(0.0, h) - 0.0) / h
    print(f"h = {h:.0e}   (f(h,0)-f(0,0))/h = {qx:.1e}   (f(0,h)-f(0,0))/h = {qy:.1e}")
    assert qx == 0.0 and qy == 0.0
# Off the axes the same function has no directional derivative along y = x:
t = np.array([1e-2, 1e-4, 1e-6])
print("f(t,t)/t =", f(t, t) / t, " -> constant 1, so D_(1,1) f(0,0) = 1 but the map is not linear")

h = 1e-01   (f(h,0)-f(0,0))/h = 0.0e+00   (f(0,h)-f(0,0))/h = 0.0e+00
h = 1e-03   (f(h,0)-f(0,0))/h = 0.0e+00   (f(0,h)-f(0,0))/h = 0.0e+00
h = 1e-05   (f(h,0)-f(0,0))/h = 0.0e+00   (f(0,h)-f(0,0))/h = 0.0e+00
h = 1e-07   (f(h,0)-f(0,0))/h = 0.0e+00   (f(0,h)-f(0,0))/h = 0.0e+00
f(t,t)/t = [1. 1. 1.]  -> constant 1, so D_(1,1) f(0,0) = 1 but the map is not linear


Both axis quotients are the exact float `0.0`, so the partials exist. The last line shows the directional derivative along `y = x` is `1`, which is why partial existence says nothing about the other directions.

### Problem L0.4 — Total Differential of a Cubic
**Source:** Adapted from Apostol, *Calculus Vol. II*, Ch. 8.

**Statement**
Find the total differential $df$ of $f(x,y) = x^2 y + 3 y^3$ at the point $(x,y) = (1, 2)$ and evaluate $df(1,2)$ for increments $dx = 0.03, dy = -0.02$.

**Intuition**

The total differential $df = f_x dx + f_y dy$ represents the principal linear change in $f$ corresponding to differential increments $dx$ and $dy$ in the independent variables.

**Solution**

1. **Compute Partial Derivatives**:

$$
   f_x(x,y) = \frac{\partial}{\partial x}(x^2 y + 3 y^3) = 2 x y
$$

$$
   f_y(x,y) = \frac{\partial}{\partial y}(x^2 y + 3 y^3) = x^2 + 9 y^2
$$

2. **Evaluate Partials at $(1, 2)$**:

$$
   f_x(1, 2) = 2(1)(2) = 4
$$

$$
   f_y(1, 2) = 1^2 + 9(2^2) = 1 + 36 = 37
$$

3. **Formulate Total Differential**:

$$
   df = 4 dx + 37 dy
$$

4. **Evaluate for $dx = 0.03$ and $dy = -0.02$**:

$$
   df(1,2)(0.03, -0.02) = 4(0.03) + 37(-0.02) = 0.12 - 0.74 = -0.62
$$

$$
\boxed{df = 4 dx + 37 dy; \quad df(1,2)(0.03, -0.02) = -0.62.}
$$

**Key takeaway** — The total differential linearizes the function at a point: $df = f_x dx + f_y dy$.

Recomputing Problem L0.4 from scratch, to check the boxed answer.

In [5]:
f = x**2 * y + 3 * y**3
fx, fy = sp.diff(f, x), sp.diff(f, y)
fx0, fy0 = fx.subs({x: 1, y: 2}), fy.subs({x: 1, y: 2})
df = fx0 * sp.Rational(3, 100) + fy0 * sp.Rational(-2, 100)
print("f_x =", fx, "  f_y =", fy)
print("f_x(1,2) =", fx0, "  f_y(1,2) =", fy0, "  df =", df, "=", float(df))
assert (fx0, fy0) == (4, 37) and df == sp.Rational(-31, 50)
# How good is the linear model?  Compare with the true increment.
exact = float(f.subs({x: sp.Rational(103, 100), y: sp.Rational(198, 100)}) - f.subs({x: 1, y: 2}))
print(f"exact increment {exact:+.6f}   linear model {float(df):+.6f}   error {abs(exact-float(df)):.2e}")

f_x = 2*x*y   f_y = x**2 + 9*y**2
f_x(1,2) = 4   f_y(1,2) = 37   df = -31/50 = -0.62
exact increment -0.612242   linear model -0.620000   error 7.76e-03


`df = -0.62` matches the boxed value. The exact increment is `-0.612242`, so the linear model is off by `7.8e-03` — second order in the increment, exactly as the remainder term predicts.

### Problem L0.5 — Clairaut on a Smooth Function
**Source:** Adapted from Marsden & Tromba, *Vector Calculus*, Ch. 2.

**Statement**
Compute $f_{xy}$ and $f_{yx}$ for $f(x,y) = x^3 y^2 + \sin(x y)$ and verify that Clairaut's theorem holds.

**Intuition**

Clairaut's theorem guarantees $f_{xy} = f_{yx}$ whenever $f_{xy}$ and $f_{yx}$ are continuous. Polynomial and trigonometric compositions are $C^\infty$ (infinitely differentiable), so equality must hold.

**Solution**

1. **Compute First Partials**:

$$
   f_x = 3 x^2 y^2 + y \cos(x y)
$$

$$
   f_y = 2 x^3 y + x \cos(x y)
$$

2. **Compute Mixed Partial $f_{xy} = \frac{\partial}{\partial y}(f_x)$**:

$$
   f_{xy} = \frac{\partial}{\partial y} [3 x^2 y^2 + y \cos(x y)] = 6 x^2 y + \left[ 1 \cdot \cos(x y) + y (-\sin(x y) \cdot x) \right] = 6 x^2 y + \cos(x y) - x y \sin(x y)
$$

3. **Compute Mixed Partial $f_{yx} = \frac{\partial}{\partial x}(f_y)$**:

$$
   f_{yx} = \frac{\partial}{\partial x} [2 x^3 y + x \cos(x y)] = 6 x^2 y + \left[ 1 \cdot \cos(x y) + x (-\sin(x y) \cdot y) \right] = 6 x^2 y + \cos(x y) - x y \sin(x y)
$$

4. **Comparison**:
   Both expressions are identical everywhere in $\mathbb{R}^2$, confirming $f_{xy} = f_{yx}$.

$$
\boxed{f_{xy} = f_{yx} = 6 x^2 y + \cos(x y) - x y \sin(x y).}
$$

**Key takeaway** — Smooth functions ($C^2$) always have commuting mixed partial derivatives.

Recomputing Problem L0.5 from scratch, to check the boxed answer.

In [6]:
f = x**3 * y**2 + sp.sin(x * y)
fxy = sp.simplify(sp.diff(f, x, y))
fyx = sp.simplify(sp.diff(f, y, x))
claim = 6 * x**2 * y + sp.cos(x * y) - x * y * sp.sin(x * y)
print("f_xy =", fxy)
print("f_yx =", fyx)
assert sp.simplify(fxy - fyx) == 0
assert sp.simplify(fxy - claim) == 0
print("f_xy - f_yx =", sp.simplify(fxy - fyx), "  and both match the boxed formula")

f_xy = 6*x**2*y - x*y*sin(x*y) + cos(x*y)
f_yx = 6*x**2*y - x*y*sin(x*y) + cos(x*y)
f_xy - f_yx = 0   and both match the boxed formula


SymPy returns literally the same expression for both orders, and the difference simplifies to `0`. Schwarz's theorem is not being tested here so much as illustrated: this `f` is `C^infinity`.

### Problem L0.6 — Filling a Removable Discontinuity
**Source:** Adapted from Marsden & Tromba, *Vector Calculus*, Ch. 2.

**Statement**
Determine the value $c \in \mathbb{R}$ such that the function:

$$
f(x,y) = \begin{cases} \frac{\sin(x^2 + y^2)}{x^2 + y^2}, & (x,y) \neq (0,0) \\ c, & (x,y) = (0,0) \end{cases}
$$

is continuous at $(0,0)$.

**Intuition**

Continuity at $(0,0)$ requires $c = f(0,0) = \lim_{(x,y)\to(0,0)} f(x,y)$.

**Solution**

1. **Set up Limit using Radial Substitution**:
   Let $u = x^2 + y^2$. As $(x,y) \to (0,0)$, $u \to 0^+$.

$$
   \lim_{(x,y)\to(0,0)} \frac{\sin(x^2 + y^2)}{x^2 + y^2} = \lim_{u \to 0^+} \frac{\sin u}{u}
$$

2. **Evaluate 1D Limit**:
   By L'Hôpital's rule or standard Taylor expansion $\sin u = u - \frac{u^3}{6} + \mathcal{O}(u^5)$:

$$
   \lim_{u \to 0^+} \frac{\sin u}{u} = 1
$$

3. **Set $c$**:
   For $f$ to be continuous at $(0,0)$, we must set $c = 1$.

$$
\boxed{c = 1.}
$$

**Key takeaway** — Symmetric functions of $x^2+y^2$ can be analyzed by single-variable substitution $u = x^2+y^2$.

Recomputing Problem L0.6 from scratch, to check the boxed answer.

In [7]:
u = sp.Symbol("u", positive=True)
print("symbolic limit of sin(u)/u as u -> 0+ :", sp.limit(sp.sin(u) / u, u, 0, "+"))
r = np.array([1e-1, 1e-2, 1e-3, 1e-4])
th = rng.uniform(0.0, 2 * np.pi, (r.size, 4000))
X, Y = r[:, None] * np.cos(th), r[:, None] * np.sin(th)
s = X**2 + Y**2
vals = np.sin(s) / s
for ri, row in zip(r, vals):
    print(f"r = {ri:.0e}   max |f - 1| over 4000 angles = {np.abs(row - 1.0).max():.3e}")
assert np.abs(vals[-1] - 1.0).max() < 1e-8

symbolic limit of sin(u)/u as u -> 0+ : 1
r = 1e-01   max |f - 1| over 4000 angles = 1.667e-05
r = 1e-02   max |f - 1| over 4000 angles = 1.667e-09
r = 1e-03   max |f - 1| over 4000 angles = 1.666e-13
r = 1e-04   max |f - 1| over 4000 angles = 0.000e+00


The symbolic limit is `1`, and the sampled values converge to `1` uniformly in the angle: the deviation falls by four orders of magnitude for each factor of ten in `r`, the `u^2/6` term of the Taylor series.

### Problem L0.7 — A Directional Derivative at a Point
**Source:** Adapted from Marsden & Tromba, *Vector Calculus*, Ch. 2.

**Statement**
Compute the directional derivative of $f(x,y,z) = x^2 y z^3$ at point $P(1, 1, 1)$ in the direction of vector $w = (2, -1, 2)$.

**Intuition**

For a differentiable function, the directional derivative is $D_v f(P) = \nabla f(P) \cdot v$, where $v = \frac{w}{\|w\|}$ is the **unit vector** in the direction of $w$.

**Solution**

1. **Normalize Vector $w$**:

$$
   \|w\| = \sqrt{2^2 + (-1)^2 + 2^2} = \sqrt{4 + 1 + 4} = \sqrt{9} = 3
$$

   Unit vector $v$:

$$
   v = \frac{w}{\|w\|} = \left(\frac{2}{3}, -\frac{1}{3}, \frac{2}{3}\right)
$$

2. **Compute Gradient $\nabla f$**:

$$
   \nabla f(x,y,z) = \left( \frac{\partial f}{\partial x}, \frac{\partial f}{\partial y}, \frac{\partial f}{\partial z} \right) = (2 x y z^3, x^2 z^3, 3 x^2 y z^2)
$$

3. **Evaluate Gradient at $P(1,1,1)$**:

$$
   \nabla f(1,1,1) = (2(1)(1)(1), 1^2(1)^3, 3(1)^2(1)(1)^2) = (2, 1, 3)
$$

4. **Compute Dot Product $\nabla f(P) \cdot v$**:

$$
   D_v f(1,1,1) = (2, 1, 3) \cdot \left(\frac{2}{3}, -\frac{1}{3}, \frac{2}{3}\right) = 2\left(\frac{2}{3}\right) + 1\left(-\frac{1}{3}\right) + 3\left(\frac{2}{3}\right) = \frac{4 - 1 + 6}{3} = \frac{9}{3} = 3
$$

$$
\boxed{D_v f(1,1,1) = 3.}
$$

**Key takeaway** — Always normalize the direction vector to unit length ($\|v\| = 1$) before taking the dot product with the gradient.

Recomputing Problem L0.7 from scratch, to check the boxed answer.

In [8]:
f = x**2 * y * z**3
grad = sp.Matrix([sp.diff(f, v) for v in (x, y, z)])
g0 = grad.subs({x: 1, y: 1, z: 1})
w = sp.Matrix([2, -1, 2])
v = w / w.norm()
Dv = (g0.T * v)[0]
print("grad f =", grad.T, "   grad f(1,1,1) =", g0.T, "   ||w|| =", w.norm(), "   D_v f =", Dv)
assert w.norm() == 3 and Dv == 3
# Using the un-normalised w would have given 9 -- three times too large.
print("grad f(1,1,1) . w (WRONG, w is not a unit vector) =", (g0.T * w)[0])

grad f = Matrix([[2*x*y*z**3, x**2*z**3, 3*x**2*y*z**2]])    grad f(1,1,1) = Matrix([[2, 1, 3]])    ||w|| = 3    D_v f = 3
grad f(1,1,1) . w (WRONG, w is not a unit vector) = 9


`D_v f(1,1,1) = 3` exactly. The last line is the classic error: dotting with the un-normalised `w` gives `9`, three times too large, because `||w|| = 3`.

### Problem L0.8 — Gâteaux versus Fréchet
**Source:** Adapted from Spivak, *Calculus on Manifolds*, Ch. 2.

**Statement**
Explain the foundational mathematical distinction between Gâteaux differentiability (directional differentiability) and Fréchet differentiability (total differentiability).

**Intuition**

Gâteaux differentiability requires the existence of scalar directional limits $D_v f(a) = \lim_{t\to 0} \frac{f(a+tv)-f(a)}{t}$ along individual rays. Fréchet differentiability requires a single linear map $L$ that approximates $f$ simultaneously across all directions in a full neighborhood of $a$ with uniform sublinear remainder.

**Solution**

1. **Gâteaux Differentiability**:
   - Focuses on 1D slices $t \mapsto f(a + t v)$.
   - Does NOT require $D_v f(a)$ to be linear in $v$.
   - Does NOT guarantee continuity of $f$ at $a$.

2. **Fréchet Differentiability**:
   - Requires $f(a+h) = f(a) + L(h) + o(\|h\|)$, where $L$ is a bounded linear map.
   - Forces $D_v f(a) = L(v)$, which is automatically linear in $v$.
   - Guarantees continuity of $f$ at $a$.

$$
\boxed{\text{Gâteaux evaluates 1D rays individually; Fréchet requires a single linear map approximating } f \text{ in all directions simultaneously with } o(\|h\|) \text{ error.}}
$$

**Key takeaway** — Fréchet differentiability is a global linear approximation condition in a multidimensional neighborhood, whereas Gâteaux differentiability is a collection of 1D directional derivative limits.

## L1 — Foundations

### Problem L1.1 — Domain and Topology of a Log-Root Function
**Source:** Adapted from Marsden & Tromba, *Vector Calculus*, 6th Edition, Ch. 2.

**Statement**
Determine the maximal domain $D \subset \mathbb{R}^2$ of the function:

$$
f(x,y) = \ln(y - x^2) + \sqrt{1 - x^2 - y^2}
$$

Classify $D$ in terms of topology (open, closed, bounded, compact).

**Intuition**

The domain of a multivariable real function is the intersection of the domains of each constituent term. The natural logarithm requires a strictly positive argument ($y - x^2 \gt 0$), while the real square root requires a non-negative argument ($1 - x^2 - y^2 \ge 0$).

**Solution**

1. **Constraint from Logarithm**:

$$
   y - x^2 \gt 0 \iff y \gt x^2
$$

   This represents the region strictly above the parabola $y = x^2$ (excluding the parabolic boundary itself).

2. **Constraint from Square Root**:

$$
   1 - x^2 - y^2 \ge 0 \iff x^2 + y^2 \le 1
$$

   This represents the closed unit disk centered at the origin, including its boundary circle $x^2 + y^2 = 1$.

3. **Intersection of Domains**:

$$
   D = \{(x,y) \in \mathbb{R}^2 \mid y \gt x^2 \text{ and } x^2 + y^2 \le 1\}
$$

4. **Topological Classification**:
   - **Open/Closed**: $D$ is neither open (contains boundary points where $x^2+y^2=1$) nor closed (does not contain boundary points along $y=x^2$).
   - **Boundedness**: $D \subset B_1(0,0)$ (contained in the unit disk), so $D$ is **bounded**.
   - **Compactness**: A subset of $\mathbb{R}^2$ is compact if and only if it is closed and bounded (Heine-Borel). Since $D$ is not closed, $D$ is **not compact**.

$$
\boxed{D = \{(x,y) \in \mathbb{R}^2 \mid y \gt x^2 \text{ and } x^2 + y^2 \le 1\}; \text{ Bounded, neither open nor closed, not compact.}}
$$

**Key takeaway** — Domain determination relies on simultaneous inequalities. Open constraints ($\gt$) yield open regions; non-strict constraints ($\ge$) include boundaries. A set is neither open nor closed if it contains some but not all of its boundary points.

Recomputing Problem L1.1 from scratch, to check the boxed answer.

In [9]:
inD = lambda P: (P[:, 1] > P[:, 0] ** 2) & (P[:, 0] ** 2 + P[:, 1] ** 2 <= 1.0)
probe = np.array([[0.0, 1.0], [0.0, 0.0], [0.0, 0.5], [0.6, 0.8], [0.9, 0.4], [0.0, 1.2]])
label = ["on x^2+y^2=1", "on y=x^2", "interior", "on x^2+y^2=1", "below y=x^2", "outside disk"]
for p, m, lab in zip(probe, inD(probe), label):
    print(f"({p[0]:5.2f},{p[1]:5.2f})  {lab:16s}  in D: {bool(m)}")
# (0,1) is a boundary point that D contains  -> D is not open.
# (0,0) is a boundary point that D omits     -> D is not closed, hence not compact.
assert inD(probe)[0] and not inD(probe)[1]
th = rng.uniform(0.0, 2 * np.pi, 200000)
rad = np.sqrt(rng.uniform(0.0, 1.0, 200000))
S = np.column_stack([rad * np.cos(th), rad * np.sin(th)])
S = S[inD(S)]
print(f"sampled {S.shape[0]} points of D; max ||(x,y)|| = {np.linalg.norm(S, axis=1).max():.6f} <= 1  -> bounded")
assert np.linalg.norm(S, axis=1).max() <= 1.0

( 0.00, 1.00)  on x^2+y^2=1      in D: True
( 0.00, 0.00)  on y=x^2          in D: False
( 0.00, 0.50)  interior          in D: True
( 0.60, 0.80)  on x^2+y^2=1      in D: True
( 0.90, 0.40)  below y=x^2       in D: False
( 0.00, 1.20)  outside disk      in D: False
sampled 67921 points of D; max ||(x,y)|| = 0.999996 <= 1  -> bounded


`(0,1)` lies on the circle and belongs to `D`, so `D` is not open; `(0,0)` lies on the parabola and does not, so `D` is not closed and therefore not compact. Every sampled point has norm at most `1`, confirming boundedness.

### Problem L1.2 — An Epsilon-Delta Proof for an Affine Limit
**Source:** Adapted from Apostol, *Mathematical Analysis*, Ch. 4.

**Statement**
Prove that $\lim_{(x,y) \to (1,2)} (3x + 2y) = 7$ using the formal $\epsilon$-$\delta$ definition of multivariable limits.

**Intuition**

We must show that for any given $\epsilon \gt 0$, we can find $\delta \gt 0$ such that $0 \lt \sqrt{(x-1)^2 + (y-2)^2} \lt \delta$ guarantees $\lvert (3x + 2y) - 7 \rvert \lt \epsilon$.

**Solution**

1. **Analyze Difference**:

$$
   \lvert (3x + 2y) - 7 \rvert = \lvert 3(x - 1) + 2(y - 2) \rvert
$$

2. **Apply Triangle Inequality**:

$$
   \lvert 3(x - 1) + 2(y - 2) \rvert \le 3\lvert x - 1 \rvert + 2\lvert y - 2 \rvert
$$

3. **Bound by Euclidean Norm**:
   Recall $\lvert x - 1 \rvert = \sqrt{(x-1)^2} \le \sqrt{(x-1)^2 + (y-2)^2}$ and $\lvert y - 2 \rvert \le \sqrt{(x-1)^2 + (y-2)^2}$.
   Let $r = \sqrt{(x-1)^2 + (y-2)^2}$. Then:

$$
   3\lvert x - 1 \rvert + 2\lvert y - 2 \rvert \le 3 r + 2 r = 5 r
$$

4. **Choose $\delta$**:
   We want $5 r \lt \epsilon$, so choose $\delta = \frac{\epsilon}{5}$.
5. **Formal Proof**:
   Given $\epsilon \gt 0$, set $\delta = \frac{\epsilon}{5}$. If $0 \lt \sqrt{(x-1)^2 + (y-2)^2} \lt \delta$, then:

$$
   \lvert (3x + 2y) - 7 \rvert \le 5 \sqrt{(x-1)^2 + (y-2)^2} \lt 5 \delta = 5 \left(\frac{\epsilon}{5}\right) = \epsilon
$$

   Hence $\lim_{(x,y)\to(1,2)} (3x + 2y) = 7$. $\blacksquare$

$$
\boxed{\text{Proof complete with } \delta = \frac{\epsilon}{5}.}
$$

**Key takeaway** — Using the triangle inequality and bounding component differences by the Euclidean distance $\|x - a\|$ yields direct $\epsilon$-$\delta$ proofs for linear multivariable functions.

Recomputing Problem L1.2 from scratch, to check the boxed answer.

In [10]:
for eps in (1e-2, 1e-4, 1e-6):
    delta = eps / 5.0
    th = rng.uniform(0.0, 2 * np.pi, 200000)
    r = delta * np.sqrt(rng.uniform(0.0, 1.0, 200000))
    X, Y = 1.0 + r * np.cos(th), 2.0 + r * np.sin(th)
    err = np.abs(3 * X + 2 * Y - 7.0)
    print(f"eps = {eps:.0e}   delta = {delta:.0e}   max |3x+2y-7| over the punctured ball = {err.max():.3e}")
    assert err.max() < eps
# A larger delta breaks the guarantee: delta = eps exceeds the bound at the boundary.
X, Y = 1.0 + 1e-2, 2.0 + 1e-2
print(f"delta = eps = 1e-2 at (x,y) = ({X}, {Y}) gives |3x+2y-7| = {abs(3*X+2*Y-7):.3e} > 1e-2")

eps = 1e-02   delta = 2e-03   max |3x+2y-7| over the punctured ball = 7.208e-03
eps = 1e-04   delta = 2e-05   max |3x+2y-7| over the punctured ball = 7.208e-05
eps = 1e-06   delta = 2e-07   max |3x+2y-7| over the punctured ball = 7.210e-07
delta = eps = 1e-2 at (x,y) = (1.01, 2.01) gives |3x+2y-7| = 5.000e-02 > 1e-2


For each `epsilon`, the measured maximum over a ball of radius `epsilon/5` is about `0.72 * epsilon`, comfortably below `epsilon`. The last line shows that `delta = epsilon` would fail: the bound `5 delta` is what makes the choice work.

### Problem L1.3 — Partials Exist, Continuity Fails
**Source:** Demidovich, *Problems in Mathematical Analysis*, No. 3182.

**Statement**
Define $f(x,y) = \frac{xy}{x^2+y^2}$ for $(x,y) \neq (0,0)$ and $f(0,0) = 0$. Show that $f_x(0,0)$ and $f_y(0,0)$ both exist, yet $f$ is discontinuous at $(0,0)$.

**Intuition**

This problem demonstrates that the existence of partial derivatives along coordinate axes is a very weak property: it only checks behavior along two orthogonal 1D lines, ignoring all intermediate directions in $\mathbb{R}^2$.

**Solution**

1. **Compute Partial Derivatives at $(0,0)$**:

$$
   f_x(0,0) = \lim_{h \to 0} \frac{f(h,0) - f(0,0)}{h} = \lim_{h \to 0} \frac{0 - 0}{h} = 0
$$

$$
   f_y(0,0) = \lim_{k \to 0} \frac{f(0,k) - f(0,0)}{k} = \lim_{k \to 0} \frac{0 - 0}{k} = 0
$$

   Both partial derivatives exist and equal $0$.

2. **Check Continuity at $(0,0)$**:
   Continuity requires $\lim_{(x,y)\to(0,0)} f(x,y) = f(0,0) = 0$.
   Approach along the line $y = x$:

$$
   f(x,x) = \frac{x^2}{x^2+x^2} = \frac{1}{2}
$$

   Thus $\lim_{x \to 0} f(x,x) = \frac{1}{2} \neq 0 = f(0,0)$.

3. **Conclusion**:
   $f$ is not continuous at $(0,0)$, despite both partial derivatives existing at $(0,0)$.

$$
\boxed{f_x(0,0) = 0, f_y(0,0) = 0; \text{ Discontinuous because } \lim_{(x,y)\to(0,0)} f(x,y) \text{ does not exist.}}
$$

**Key takeaway** — The existence of partial derivatives does NOT imply continuity in multivariable calculus.

Recomputing Problem L1.3 from scratch, to check the boxed answer.

In [11]:
g = lambda X, Y: X * Y / (X**2 + Y**2)
t = np.array([1e-2, 1e-4, 1e-6, 1e-8])
print("f(h,0)/h  =", np.zeros_like(t), "  (f vanishes on the x-axis, so f_x(0,0) = 0)")
print("f(0,k)/k  =", np.zeros_like(t), "  (f vanishes on the y-axis, so f_y(0,0) = 0)")
print("f(t,t)    =", g(t, t), "  -> 1/2, not f(0,0) = 0")
assert np.allclose(g(t, t), 0.5) and np.allclose(g(t, np.zeros_like(t)), 0.0)
# The values of f on a shrinking circle do not concentrate: the whole range [-1/2, 1/2] survives.
th = np.linspace(0.0, 2 * np.pi, 2001)
for r in (1e-2, 1e-6):
    v = g(r * np.cos(th), r * np.sin(th))
    print(f"r = {r:.0e}   range of f on the circle = [{v.min():+.4f}, {v.max():+.4f}]")

f(h,0)/h  = [0. 0. 0. 0.]   (f vanishes on the x-axis, so f_x(0,0) = 0)
f(0,k)/k  = [0. 0. 0. 0.]   (f vanishes on the y-axis, so f_y(0,0) = 0)
f(t,t)    = [0.5 0.5 0.5 0.5]   -> 1/2, not f(0,0) = 0
r = 1e-02   range of f on the circle = [-0.5000, +0.5000]
r = 1e-06   range of f on the circle = [-0.5000, +0.5000]


The function is identically zero on both axes, so both partials are `0`; but on the diagonal it is pinned at `1/2`, and its range on every circle around the origin is the full interval `[-1/2, 1/2]`, no matter how small the radius.

### Problem L1.4 — Lines Are Not Enough
**Source:** Demidovich, *Problems in Mathematical Analysis*, No. 3182.

**Statement**
Show that $\lim_{(x,y)\to(0,0)} \frac{x^2 y}{x^4 + y^2}$ does not exist, even though the limit along every straight line $y = m x$ is equal to 0.

**Intuition**

Linear paths $y = mx$ approach the origin along lines. However, the denominator $x^4 + y^2$ balances when $y$ is of order $x^2$. Testing parabolic paths $y = k x^2$ probes the true geometry of the singularity.

**Solution**

1. **Straight-Line Paths $y = m x$**:
   For $m \neq 0$:

$$
   f(x, mx) = \frac{x^2 (mx)}{x^4 + m^2 x^2} = \frac{m x^3}{x^2(x^2 + m^2)} = \frac{m x}{x^2 + m^2} \xrightarrow{x \to 0} 0
$$

   For $m = 0$ ($y=0$), $f(x,0) = 0 \to 0$. Thus the limit along all straight lines is 0.

2. **Parabolic Paths $y = k x^2$**:
   Substitute $y = k x^2$ for $x \neq 0$:

$$
   f(x, k x^2) = \frac{x^2 (k x^2)}{x^4 + (k x^2)^2} = \frac{k x^4}{x^4 + k^2 x^4} = \frac{k x^4}{x^4(1 + k^2)} = \frac{k}{1 + k^2}
$$

3. **Evaluate Path Limit**:

$$
   \lim_{x \to 0} f(x, k x^2) = \frac{k}{1 + k^2}
$$

   - For $k = 1$ ($y = x^2$), the limit is $\frac{1}{2}$.
   - For $k = -1$ ($y = -x^2$), the limit is $-\frac{1}{2}$.

4. **Conclusion**:
   Since the limit along $y = x^2$ ($\frac{1}{2}$) differs from the limit along straight lines ($0$), the multivariable limit **does not exist**.

$$
\boxed{\text{Limit along } y=kx^2 \text{ is } \frac{k}{1+k^2}; \text{ Limit does not exist.}}
$$

**Key takeaway** — Straight line approaches are insufficient to establish multivariable limits. Parabolic and higher-order curves must be checked when degree imbalances occur in denominators.

Recomputing Problem L1.4 from scratch, to check the boxed answer.

In [12]:
f = lambda X, Y: X**2 * Y / (X**4 + Y**2)
t = np.array([1e-2, 1e-3, 1e-4, 1e-5])
for m in (1.0, -2.0, 5.0):
    print(f"line     y = {m:+.1f} x     f -> {f(t, m*t)}")
    assert np.abs(f(t, m * t)).max() < 1e-1
for k in (1.0, -1.0, 3.0):
    print(f"parabola y = {k:+.1f} x^2   f  = {f(t, k*t**2)}   predicted k/(1+k^2) = {k/(1+k**2):+.4f}")
    assert np.allclose(f(t, k * t**2), k / (1 + k**2))

line     y = +1.0 x     f -> [0.009999 0.001    0.0001   0.00001 ]
line     y = -2.0 x     f -> [-0.005    -0.0005   -0.00005  -0.000005]
line     y = +5.0 x     f -> [0.002    0.0002   0.00002  0.000002]
parabola y = +1.0 x^2   f  = [0.5 0.5 0.5 0.5]   predicted k/(1+k^2) = +0.5000
parabola y = -1.0 x^2   f  = [-0.5 -0.5 -0.5 -0.5]   predicted k/(1+k^2) = -0.5000
parabola y = +3.0 x^2   f  = [0.3 0.3 0.3 0.3]   predicted k/(1+k^2) = +0.3000


The line values shrink like `x`, while the parabola values sit at `k/(1+k^2)` for every `x`. Lines detect nothing here because the singularity lives at the scale `y ~ x^2`.

### Problem L1.5 — A Polar-Coordinate Squeeze
**Source:** Adapted from Demidovich, *Problems in Mathematical Analysis*, No. 3186.

**Statement**
Evaluate the limit using polar coordinates:

$$
L = \lim_{(x,y)\to(0,0)} \frac{x^3 + y^3}{x^2 + y^2}
$$

**Intuition**

When a limit involves $x^2 + y^2$ in the denominator, converting to polar coordinates ($x = r \cos\theta, y = r \sin\theta$) separates the radial distance $r \to 0^+$ from the directional angle $\theta \in [0, 2\pi)$. If the resulting expression can be bounded by a function of $r$ independent of $\theta$, the limit exists.

**Solution**

1. **Substitute Polar Coordinates**:
   Let $x = r \cos\theta$ and $y = r \sin\theta$. As $(x,y) \to (0,0)$, $r \to 0^+$ while $\theta$ is arbitrary.

$$
   f(r\cos\theta, r\sin\theta) = \frac{r^3 \cos^3\theta + r^3 \sin^3\theta}{r^2 \cos^2\theta + r^2 \sin^2\theta} = \frac{r^3 (\cos^3\theta + \sin^3\theta)}{r^2} = r (\cos^3\theta + \sin^3\theta)
$$

2. **Bound the Expression**:
   Since $\lvert\cos\theta\rvert \le 1$ and $\lvert\sin\theta\rvert \le 1$:

$$
   \lvert\cos^3\theta + \sin^3\theta\rvert \le \lvert\cos\theta\rvert^3 + \lvert\sin\theta\rvert^3 \le 1 + 1 = 2
$$

   Therefore:

$$
   \lvert f(r\cos\theta, r\sin\theta) \rvert \le 2 r
$$

3. **Apply Squeeze Theorem**:
   As $r \to 0^+$, $2 r \to 0$. Since $-2r \le f(r\cos\theta, r\sin\theta) \le 2r$, by the Squeeze Theorem:

$$
   \lim_{r \to 0^+} f(r\cos\theta, r\sin\theta) = 0
$$

   Thus $L = 0$.

$$
\boxed{L = 0.}
$$

**Key takeaway** — Polar coordinate transformation reduces 2D limits to 1D radial limits $r \to 0^+$. Uniform boundedness with respect to $\theta$ guarantees existence of the limit.

Recomputing Problem L1.5 from scratch, to check the boxed answer.

In [13]:
r = np.array([1e-1, 1e-2, 1e-3, 1e-4])
th = rng.uniform(0.0, 2 * np.pi, (r.size, 4000))
X, Y = r[:, None] * np.cos(th), r[:, None] * np.sin(th)
vals = (X**3 + Y**3) / (X**2 + Y**2)
for ri, row in zip(r, vals):
    print(f"r = {ri:.0e}   max |f| over 4000 angles = {np.abs(row).max():.3e}   bound 2r = {2*ri:.1e}")
assert np.all(np.abs(vals) <= 2 * r[:, None] + 1e-15)
assert np.abs(vals[-1]).max() < 1e-3

r = 1e-01   max |f| over 4000 angles = 1.000e-01   bound 2r = 2.0e-01
r = 1e-02   max |f| over 4000 angles = 1.000e-02   bound 2r = 2.0e-02
r = 1e-03   max |f| over 4000 angles = 1.000e-03   bound 2r = 2.0e-03
r = 1e-04   max |f| over 4000 angles = 1.000e-04   bound 2r = 2.0e-04


The measured maximum over 4000 random angles equals `r` at every radius, safely inside the proved bound `2r`. The squeeze is uniform in `theta`, which is exactly what makes the limit exist.

### Problem L1.6 — A Third-Order Mixed Partial
**Source:** Adapted from Stewart, *Multivariable Calculus*, Ch. 14.

**Statement**
For $f(x,y,z) = e^{x y z}$, compute the third-order mixed partial derivative $\frac{\partial^3 f}{\partial x \partial y \partial z}$.

**Intuition**

Higher-order partial derivatives are computed by applying standard single-variable derivative rules sequentially with respect to each designated variable.

**Solution**

1. **First Partial w.r.t. $z$**:

$$
   f_z = \frac{\partial}{\partial z}(e^{x y z}) = x y e^{x y z}
$$

2. **Second Partial w.r.t. $y$**:
   Apply product rule to $(x y)(e^{x y z})$:

$$
   f_{zy} = \frac{\partial}{\partial y}(x y e^{x y z}) = x e^{x y z} + x y (x z e^{x y z}) = (x + x^2 y z) e^{x y z}
$$

3. **Third Partial w.r.t. $x$**:
   Apply product rule to $(x + x^2 y z) e^{x y z}$:

$$
   \begin{aligned}
   f_{zyx} &= \frac{\partial}{\partial x} \left[ (x + x^2 y z) e^{x y z} \right] \\
   &= (1 + 2 x y z) e^{x y z} + (x + x^2 y z) (y z e^{x y z}) \\
   &= \left( 1 + 2 x y z + x y z + x^2 y^2 z^2 \right) e^{x y z} \\
   &= (1 + 3 x y z + x^2 y^2 z^2) e^{x y z}
   \end{aligned}
$$

$$
\boxed{\frac{\partial^3 f}{\partial x \partial y \partial z} = (1 + 3 x y z + x^2 y^2 z^2) e^{x y z}.}
$$

**Key takeaway** — Sequential differentiation requires careful application of the product rule and chain rule at each step.

Recomputing Problem L1.6 from scratch, to check the boxed answer.

In [14]:
f = sp.exp(x * y * z)
claim = (1 + 3 * x * y * z + x**2 * y**2 * z**2) * sp.exp(x * y * z)
print("d^3 f / dx dy dz =", sp.simplify(sp.diff(f, z, y, x)))
assert sp.simplify(sp.diff(f, z, y, x) - claim) == 0
for order in itertools.permutations((x, y, z)):
    assert sp.simplify(sp.diff(f, *order) - claim) == 0
print("all 6 differentiation orders agree, as Schwarz's theorem predicts for this C-infinity f")
print("value at (1,1,1):", float(claim.subs({x: 1, y: 1, z: 1})), "=", float(5 * sp.E))

d^3 f / dx dy dz = (x**2*y**2*z**2 + 3*x*y*z + 1)*exp(x*y*z)


all 6 differentiation orders agree, as Schwarz's theorem predicts for this C-infinity f


value at (1,1,1): 13.591409142295227 = 13.591409142295227


All six differentiation orders return the same expression, and at `(1,1,1)` the value is `5e = 13.5914`. For a `C^infinity` function the order of differentiation is pure bookkeeping.

### Problem L1.7 — Permuting Three Differentiations
**Source:** Adapted from Stewart, *Multivariable Calculus*, Ch. 14.

**Statement**
Compute $f_{xyy}$ and $f_{yxy}$ for $f(x,y) = x \cos(y) + y e^x$ and confirm their equality.

**Intuition**

Clairaut's theorem extends to higher-order derivatives of $C^k$ functions: order of differentiation can be permuted arbitrarily.

**Solution**

1. **Compute $f_x$ and $f_y$**:

$$
   f_x = \cos(y) + y e^x
$$

$$
   f_y = -x \sin(y) + e^x
$$

2. **Compute $f_{xy} = \frac{\partial}{\partial y}(f_x)$**:

$$
   f_{xy} = -\sin(y) + e^x
$$

3. **Compute $f_{xyy} = \frac{\partial}{\partial y}(f_{xy})$**:

$$
   f_{xyy} = -\cos(y) + 0 = -\cos(y)
$$

4. **Compute $f_{yx} = \frac{\partial}{\partial x}(f_y)$**:

$$
   f_{yx} = -\sin(y) + e^x
$$

5. **Compute $f_{yxy} = \frac{\partial}{\partial y}(f_{yx})$**:

$$
   f_{yxy} = -\cos(y)
$$

   Both equal $-\cos(y)$.

$$
\boxed{f_{xyy} = f_{yxy} = -\cos(y).}
$$

**Key takeaway** — Permuting partial differentiation order for smooth functions yields identical results.

Recomputing Problem L1.7 from scratch, to check the boxed answer.

In [15]:
f = x * sp.cos(y) + y * sp.exp(x)
fxyy = sp.simplify(sp.diff(f, x, y, y))
fyxy = sp.simplify(sp.diff(f, y, x, y))
fyyx = sp.simplify(sp.diff(f, y, y, x))
print("f_xyy =", fxyy, "   f_yxy =", fyxy, "   f_yyx =", fyyx)
assert fxyy == fyxy == fyyx == -sp.cos(y)

f_xyy = -cos(y)    f_yxy = -cos(y)    f_yyx = -cos(y)


All three orders return `-cos(y)`. The `y e^x` term contributes `e^x` to `f_xy` and then dies under the second `y` derivative.

### Problem L1.8 — Linear Approximation of a Euclidean Norm
**Source:** Adapted from Apostol, *Calculus Vol. II*, Ch. 8.

**Statement**
Use total differential linear approximation to estimate $\sqrt{(3.02)^2 + (3.99)^2}$.

**Intuition**

We approximate $f(x+dx, y+dy) \approx f(x,y) + f_x(x,y) dx + f_y(x,y) dy$ around a convenient base point $(x_0, y_0) = (3, 4)$ where $f(3,4) = \sqrt{9+16} = 5$ is exact.

**Solution**

1. **Define Function and Base Point**:
   Let $f(x,y) = \sqrt{x^2 + y^2}$. Base point $(x_0, y_0) = (3, 4)$, so $f(3,4) = 5$.
   Increments: $dx = 0.02$, $dy = -0.01$.

2. **Compute Partials at Base Point**:

$$
   f_x(x,y) = \frac{x}{\sqrt{x^2+y^2}} \implies f_x(3,4) = \frac{3}{5} = 0.6
$$

$$
   f_y(x,y) = \frac{y}{\sqrt{x^2+y^2}} \implies f_y(3,4) = \frac{4}{5} = 0.8
$$

3. **Evaluate Total Differential**:

$$
   df = f_x(3,4) dx + f_y(3,4) dy = 0.6(0.02) + 0.8(-0.01) = 0.012 - 0.008 = 0.004
$$

4. **Linear Approximation**:

$$
   f(3.02, 3.99) \approx f(3,4) + df = 5 + 0.004 = 5.004
$$

$$
\boxed{\sqrt{(3.02)^2 + (3.99)^2} \approx 5.004.}
$$

**Key takeaway** — The linear approximation formula $f(x_0+dx, y_0+dy) \approx f(x_0, y_0) + \nabla f(x_0, y_0) \cdot (dx, dy)$ enables rapid estimation with high accuracy for small increments.

Recomputing Problem L1.8 from scratch, to check the boxed answer.

In [16]:
exact = float(np.hypot(3.02, 3.99))
linear = 5.0 + 0.6 * 0.02 + 0.8 * (-0.01)
print(f"exact  sqrt(3.02^2 + 3.99^2) = {exact:.12f}")
print(f"linear 5 + 0.6*0.02 + 0.8*(-0.01) = {linear:.12f}")
print(f"absolute error {abs(exact-linear):.3e}   relative error {abs(exact-linear)/exact:.3e}")
assert abs(exact - linear) < 5e-5
# Second-order behaviour: halving the increment quarters the error.
err = []
for s in (1.0, 0.5, 0.25, 0.125):
    dx, dy = 0.02 * s, -0.01 * s
    err.append(abs(np.hypot(3 + dx, 4 + dy) - (5.0 + 0.6 * dx + 0.8 * dy)))
print("errors:", np.array(err), "  ratios:", np.array(err[:-1]) / np.array(err[1:]), "  predicted 4")
assert np.allclose(np.array(err[:-1]) / np.array(err[1:]), 4.0, rtol=1e-2)

exact  sqrt(3.02^2 + 3.99^2) = 5.004048361077
linear 5 + 0.6*0.02 + 0.8*(-0.01) = 5.004000000000
absolute error 4.836e-05   relative error 9.664e-06
errors: [0.000048 0.000012 0.000003 0.000001]   ratios: [3.998387 3.999197 3.999599]   predicted 4


The linear estimate `5.004` is within `4.8e-05` of the true value `5.004048`. Halving the increment quarters the error — the observed ratios are `4.00` — confirming the remainder is second order.

### Problem L1.9 — Fréchet Differentiability from the Definition
**Source:** Spivak, *Calculus on Manifolds*, Ch. 2.

**Statement**
Verify that $f(x,y) = x^2 + y^2$ is Fréchet differentiable at any point $(x_0, y_0) \in \mathbb{R}^2$ by explicitly finding the linear map $L$ and proving $\lim_{h \to 0} \frac{\|E(h)\|}{\|h\|} = 0$.

**Intuition**

We expand $f(x_0+h_1, y_0+h_2) - f(x_0, y_0)$ into linear terms in $(h_1, h_2)$ and higher-order terms, then show the remainder vanishes faster than $\|h\| = \sqrt{h_1^2 + h_2^2}$.

**Solution**

1. **Expand $f(x_0+h_1, y_0+h_2)$**:

$$
   \begin{aligned}
   f(x_0+h_1, y_0+h_2) &= (x_0+h_1)^2 + (y_0+h_2)^2 \\
   &= x_0^2 + 2 x_0 h_1 + h_1^2 + y_0^2 + 2 y_0 h_2 + h_2^2 \\
   &= (x_0^2 + y_0^2) + (2 x_0 h_1 + 2 y_0 h_2) + (h_1^2 + h_2^2)
   \end{aligned}
$$

2. **Identify Candidate Linear Map $L$ and Remainder $E(h)$**:

$$
   f(x_0+h_1, y_0+h_2) - f(x_0, y_0) = \underbrace{2 x_0 h_1 + 2 y_0 h_2}_{L(h_1, h_2)} + \underbrace{(h_1^2 + h_2^2)}_{E(h_1, h_2)}
$$

   Note:

$$
   L(h_1, h_2) = \begin{bmatrix} 2 x_0 & 2 y_0 \end{bmatrix} \begin{bmatrix} h_1 \\ h_2 \end{bmatrix}
$$

   which is a valid linear operator.

3. **Verify Remainder Condition**:
   Let $\|h\| = \sqrt{h_1^2 + h_2^2}$. The remainder is $E(h_1, h_2) = h_1^2 + h_2^2 = \|h\|^2$.
   Compute the limit:

$$
   \lim_{h \to 0} \frac{\lvert E(h) \rvert}{\|h\|} = \lim_{h \to 0} \frac{\|h\|^2}{\|h\|} = \lim_{h \to 0} \|h\| = 0
$$

4. **Conclusion**:
   $f$ is Fréchet differentiable at $(x_0, y_0)$ with total derivative:

$$
   D f(x_0, y_0) = \begin{bmatrix} 2 x_0 & 2 y_0 \end{bmatrix}
$$

$$
\boxed{L(h_1, h_2) = 2 x_0 h_1 + 2 y_0 h_2; \quad \lim_{h\to 0} \frac{\|E(h)\|}{\|h\|} = 0.}
$$

**Key takeaway** — Proving Fréchet differentiability requires isolating the linear map $L(h) = \nabla f(a) \cdot h$ and demonstrating that the error $E(h) = o(\|h\|)$.

Recomputing Problem L1.9 from scratch, to check the boxed answer.

In [17]:
a = np.array([0.7, -1.3])
f = lambda p: p[0] ** 2 + p[1] ** 2
L = 2 * a                      # the claimed derivative Df(a) h = 2a . h
u = np.array([0.6, 0.8])       # a unit direction
hs = 0.1 * 0.5 ** np.arange(8)
q = np.array([abs(f(a + h * u) - f(a) - L @ (h * u)) / h for h in hs])
rates = np.log2(q[:-1] / q[1:])
print("|E(h)| / ||h|| :", q)
print("observed order :", rates, "   predicted 1.0")
assert np.allclose(rates, 1.0, atol=1e-9) and q[-1] < 1e-3
# A wrong linear map -- say 3a instead of 2a -- leaves a quotient that does not vanish.
Lbad = 3 * a
qb = np.array([abs(f(a + h * u) - f(a) - Lbad @ (h * u)) / h for h in hs])
print("with the wrong L :", qb, "  -> tends to |a.u|, not 0")

|E(h)| / ||h|| : [0.1      0.05     0.025    0.0125   0.00625  0.003125 0.001563 0.000781]
observed order : [1. 1. 1. 1. 1. 1. 1.]    predicted 1.0
with the wrong L : [0.72     0.67     0.645    0.6325   0.62625  0.623125 0.621563 0.620781]   -> tends to |a.u|, not 0


The measured order is exactly `1.0` at every refinement: `|E(h)| / ||h|| = ||h||`, so the remainder is `o(||h||)` and `L(h) = 2a . h` is the derivative. With the wrong `L` the quotient stalls at `|a . u|` instead of vanishing — the definition really does pin down a unique linear map.

### Problem L1.10 — Jacobian of Spherical Coordinates
**Source:** Adapted from Apostol, *Calculus Vol. II*, Ch. 8.

**Statement**
Compute the $3 \times 3$ Jacobian matrix $J_T(r, \theta, \phi)$ for the spherical coordinate transformation $T(r, \theta, \phi) = (x, y, z)$:

$$
x = r \sin\phi \cos\theta, \quad y = r \sin\phi \sin\theta, \quad z = r \cos\phi
$$

**Intuition**

The Jacobian matrix of a vector function $T: \mathbb{R}^3 \to \mathbb{R}^3$ organizes all partial derivatives $\frac{\partial T_i}{\partial u_j}$ into a matrix where row $i$ contains partial derivatives of the $i$-th coordinate.

**Solution**

1. **Partials of $x = r \sin\phi \cos\theta$**:

$$
   \frac{\partial x}{\partial r} = \sin\phi \cos\theta, \quad \frac{\partial x}{\partial \theta} = -r \sin\phi \sin\theta, \quad \frac{\partial x}{\partial \phi} = r \cos\phi \cos\theta
$$

2. **Partials of $y = r \sin\phi \sin\theta$**:

$$
   \frac{\partial y}{\partial r} = \sin\phi \sin\theta, \quad \frac{\partial y}{\partial \theta} = r \sin\phi \cos\theta, \quad \frac{\partial y}{\partial \phi} = r \cos\phi \sin\theta
$$

3. **Partials of $z = r \cos\phi$**:

$$
   \frac{\partial z}{\partial r} = \cos\phi, \quad \frac{\partial z}{\partial \theta} = 0, \quad \frac{\partial z}{\partial \phi} = -r \sin\phi
$$

4. **Assemble Jacobian Matrix**:

$$
   J_T(r, \theta, \phi) = \begin{bmatrix}
   \sin\phi \cos\theta & -r \sin\phi \sin\theta & r \cos\phi \cos\theta \\
   \sin\phi \sin\theta & r \sin\phi \cos\theta & r \cos\phi \sin\theta \\
   \cos\phi & 0 & -r \sin\phi
   \end{bmatrix}
$$

5. **Determinant**:
   Expanding along the bottom row,

$$
   \det J_T = \cos\phi \bigl(-r^2 \sin\phi \cos\phi\bigr) - r \sin\phi \bigl(r \sin^2\phi\bigr) = -r^2 \sin\phi (\cos^2\phi + \sin^2\phi) = -r^2 \sin\phi
$$

   The sign is an artefact of the column order $(r, \theta, \phi)$; the change-of-variables
   formula uses $\lvert \det J_T \rvert = r^2 \sin\phi$, which is positive on $0 \lt \phi \lt \pi$.

$$
\boxed{J_T(r, \theta, \phi) = \begin{bmatrix} \sin\phi \cos\theta & -r \sin\phi \sin\theta & r \cos\phi \cos\theta \\ \sin\phi \sin\theta & r \sin\phi \cos\theta & r \cos\phi \sin\theta \\ \cos\phi & 0 & -r \sin\phi \end{bmatrix}, \quad \det J_T = -r^2 \sin\phi.}
$$

**Key takeaway** — The Jacobian matrix is the best linear approximation to the coordinate change at a point; with the columns ordered $(r, \theta, \phi)$ its determinant is $-r^2 \sin\phi$, and the volume element uses its absolute value $r^2 \sin\phi$.

Recomputing Problem L1.10 from scratch, to check the boxed answer.

In [18]:
r, th, ph = sp.symbols("r theta phi", positive=True)
T = sp.Matrix([r * sp.sin(ph) * sp.cos(th), r * sp.sin(ph) * sp.sin(th), r * sp.cos(ph)])
J = sp.simplify(T.jacobian([r, th, ph]))
claim = sp.Matrix([
    [sp.sin(ph) * sp.cos(th), -r * sp.sin(ph) * sp.sin(th), r * sp.cos(ph) * sp.cos(th)],
    [sp.sin(ph) * sp.sin(th), r * sp.sin(ph) * sp.cos(th), r * sp.cos(ph) * sp.sin(th)],
    [sp.cos(ph), 0, -r * sp.sin(ph)],
])
sp.pprint(J)
assert sp.simplify(J - claim) == sp.zeros(3, 3)
det = sp.simplify(J.det())
print("det J =", det, "   |det J| =", sp.simplify(sp.Abs(det)))
assert sp.simplify(det + r**2 * sp.sin(ph)) == 0

⎡sin(φ)⋅cos(θ)  -r⋅sin(φ)⋅sin(θ)  r⋅cos(φ)⋅cos(θ)⎤
⎢                                                ⎥
⎢sin(φ)⋅sin(θ)  r⋅sin(φ)⋅cos(θ)   r⋅sin(θ)⋅cos(φ)⎥
⎢                                                ⎥
⎣   cos(φ)             0             -r⋅sin(φ)   ⎦


det J = -r**2*sin(phi)    |det J| = r**2*Abs(sin(phi))


The symbolic Jacobian matches the boxed matrix entry for entry. With the columns ordered `(r, theta, phi)` the determinant is `-r^2 sin(phi)`; the volume element uses its absolute value `r^2 sin(phi)`, which is positive on `0 < phi < pi`.

## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — The Gaussian Heat Kernel
**Source:** Physics / Heat Conduction Theory (Marsden & Tromba).

**Statement**
Verify that the fundamental Gaussian heat kernel:

$$
u(x,t) = \frac{1}{\sqrt{4\pi \alpha t}} e^{-\frac{x^2}{4\alpha t}}, \quad t \gt 0
$$

satisfies the 1D heat equation $\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}$.

**Intuition**

The heat kernel describes spatial diffusion of thermal energy over time. Verifying that $u(x,t)$ satisfies the PDE requires computing $\frac{\partial u}{\partial t}$ and $\frac{\partial^2 u}{\partial x^2}$ explicitly using the product and chain rules.

**Solution**

1. **Compute $\frac{\partial u}{\partial t}$**:
   Write $u(x,t) = (4\pi \alpha t)^{-\frac{1}{2}} e^{-\frac{x^2}{4\alpha t}}$.

$$
   \begin{aligned}
   \frac{\partial u}{\partial t} &= -\frac{1}{2} (4\pi\alpha)^{-\frac{1}{2}} t^{-\frac{3}{2}} e^{-\frac{x^2}{4\alpha t}} + (4\pi\alpha t)^{-\frac{1}{2}} e^{-\frac{x^2}{4\alpha t}} \left( \frac{x^2}{4\alpha t^2} \right) \\
   &= u(x,t) \left[ -\frac{1}{2t} + \frac{x^2}{4\alpha t^2} \right]
   \end{aligned}
$$

2. **Compute First Spatial Derivative $\frac{\partial u}{\partial x}$**:

$$
   \frac{\partial u}{\partial x} = u(x,t) \left( -\frac{2x}{4\alpha t} \right) = -\frac{x}{2\alpha t} u(x,t)
$$

3. **Compute Second Spatial Derivative $\frac{\partial^2 u}{\partial x^2}$**:

$$
   \begin{aligned}
   \frac{\partial^2 u}{\partial x^2} &= \frac{\partial}{\partial x} \left[ -\frac{x}{2\alpha t} u(x,t) \right] = -\frac{1}{2\alpha t} u(x,t) - \frac{x}{2\alpha t} \frac{\partial u}{\partial x} \\
   &= -\frac{1}{2\alpha t} u(x,t) - \frac{x}{2\alpha t} \left( -\frac{x}{2\alpha t} u(x,t) \right) \\
   &= u(x,t) \left[ -\frac{1}{2\alpha t} + \frac{x^2}{4\alpha^2 t^2} \right]
   \end{aligned}
$$

4. **Multiply by $\alpha$ and Compare**:

$$
   \alpha \frac{\partial^2 u}{\partial x^2} = \alpha u(x,t) \left[ -\frac{1}{2\alpha t} + \frac{x^2}{4\alpha^2 t^2} \right] = u(x,t) \left[ -\frac{1}{2t} + \frac{x^2}{4\alpha t^2} \right] = \frac{\partial u}{\partial t}
$$

   Thus the heat equation holds identically.

$$
\boxed{\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2} = u(x,t) \left[ -\frac{1}{2t} + \frac{x^2}{4\alpha t^2} \right].}
$$

**Key takeaway** — The Gaussian heat kernel satisfies the PDE because time-decay of peak height precisely balances spatial spreading by thermal diffusion.

Recomputing Problem L2.1 from scratch, to check the boxed answer.

In [19]:
X, T, al = sp.symbols("x t alpha", positive=True)
u = (4 * sp.pi * al * T) ** sp.Rational(-1, 2) * sp.exp(-X**2 / (4 * al * T))
residual = sp.simplify(sp.diff(u, T) - al * sp.diff(u, X, 2))
print("u_t - alpha u_xx =", residual)
assert residual == 0
print("u_t / u =", sp.simplify(sp.diff(u, T) / u))
# The kernel keeps unit mass at every time.
for tv, av in ((0.25, 1.0), (2.0, 0.3)):
    s = float(sp.integrate(u.subs({al: av, T: tv}), (X, 0, sp.oo)) * 2)
    print(f"alpha = {av}, t = {tv}:  integral of u over R = {s:.12f}")
    assert abs(s - 1.0) < 1e-12

u_t - alpha u_xx = 0
u_t / u = (-2*alpha*t + x**2)/(4*alpha*t**2)


alpha = 1.0, t = 0.25:  integral of u over R = 1.000000000000
alpha = 0.3, t = 2.0:  integral of u over R = 1.000000000000


The residual `u_t - alpha u_xx` simplifies to exactly `0` for symbolic `alpha` and `t`, and the kernel integrates to `1` at every time — it is a probability density spreading at rate `alpha`.

### Problem L2.2 — D'Alembert's Wave Solution
**Source:** Classical Physics / D'Alembert's Wave Solution (Apostol Vol. II).

**Statement**
Show that any function $u(x,t) = f(x - c t) + g(x + c t)$ with $f, g \in C^2(\mathbb{R})$ satisfies the 1D wave equation:

$$
\frac{\partial^2 u}{\partial t^2} = c^2 \frac{\partial^2 u}{\partial x^2}
$$

**Intuition**

D'Alembert's solution represents two traveling waves moving in opposite directions at speed $c$. The chain rule scales temporal derivatives by factors of $\pm c$, which square to $c^2$ in second partial derivatives.

**Solution**

1. **Define Intermediate Variables**:
   Let $\xi = x - c t$ and $\eta = x + c t$. Then $u(x,t) = f(\xi) + g(\eta)$.
2. **First Partial Derivatives**:

$$
   \frac{\partial u}{\partial x} = f'(\xi)\frac{\partial \xi}{\partial x} + g'(\eta)\frac{\partial \eta}{\partial x} = f'(\xi)(1) + g'(\eta)(1) = f'(\xi) + g'(\eta)
$$

$$
   \frac{\partial u}{\partial t} = f'(\xi)\frac{\partial \xi}{\partial t} + g'(\eta)\frac{\partial \eta}{\partial t} = f'(\xi)(-c) + g'(\eta)(c) = -c f'(\xi) + c g'(\eta)
$$

3. **Second Partial Derivatives**:

$$
   \frac{\partial^2 u}{\partial x^2} = f''(\xi)(1) + g''(\eta)(1) = f''(\xi) + g''(\eta)
$$

$$
   \frac{\partial^2 u}{\partial t^2} = -c f''(\xi)(-c) + c g''(\eta)(c) = c^2 f''(\xi) + c^2 g''(\eta) = c^2 [f''(\xi) + g''(\eta)]
$$

4. **Conclusion**:

$$
   \frac{\partial^2 u}{\partial t^2} = c^2 \frac{\partial^2 u}{\partial x^2}
$$

$$
\boxed{\frac{\partial^2 u}{\partial t^2} = c^2 (f''(\xi) + g''(\eta)) = c^2 \frac{\partial^2 u}{\partial x^2}.}
$$

**Key takeaway** — The chain rule applied to $\xi = x \mp ct$ introduces factors of $\mp c$, so second time derivatives pick up a factor of $c^2$.

Recomputing Problem L2.2 from scratch, to check the boxed answer.

In [20]:
X, T = sp.symbols("x t", real=True)
c = sp.Symbol("c", positive=True)
F, G = sp.Function("f"), sp.Function("g")
u = F(X - c * T) + G(X + c * T)
residual = sp.simplify(sp.diff(u, T, 2) - c**2 * sp.diff(u, X, 2))
print("u_tt - c^2 u_xx =", residual)
assert residual == 0
# A concrete pair, and a non-solution for contrast.
u2 = sp.sin(X - c * T) + (X + c * T) ** 3
print("concrete f, g :", sp.simplify(sp.diff(u2, T, 2) - c**2 * sp.diff(u2, X, 2)))
bad = sp.sin(X - 2 * c * T)
print("wrong speed 2c:", sp.simplify(sp.diff(bad, T, 2) - c**2 * sp.diff(bad, X, 2)))
assert sp.simplify(sp.diff(bad, T, 2) - c**2 * sp.diff(bad, X, 2)) != 0

u_tt - c^2 u_xx = 0
concrete f, g : 0


wrong speed 2c: 3*c**2*sin(2*c*t - x)


The residual vanishes for arbitrary `C^2` functions `f` and `g`, not just for a chosen pair. Substituting the wrong speed leaves `3 c^2 sin(2ct - x)`, so the factor `c` in the argument is doing real work.

### Problem L2.3 — The Logarithmic Potential Is Harmonic
**Source:** Demidovich, *Problems in Mathematical Analysis*, No. 3200.

**Statement**
Show that $u(x,y) = \ln(x^2 + y^2)$ satisfies Laplace's equation $\nabla^2 u = u_{xx} + u_{yy} = 0$ for all $(x,y) \neq (0,0)$.

**Intuition**

$u(x,y) = \ln(x^2+y^2)$ is proportional to the 2D gravitational/electrostatic potential of a point source. Outside the origin, it must be harmonic ($\nabla^2 u = 0$).

**Solution**

1. **Compute First Partials**:

$$
   u_x = \frac{\partial}{\partial x}[\ln(x^2+y^2)] = \frac{2x}{x^2+y^2}
$$

$$
   u_y = \frac{\partial}{\partial y}[\ln(x^2+y^2)] = \frac{2y}{x^2+y^2}
$$

2. **Compute $u_{xx}$ using Quotient Rule**:

$$
   u_{xx} = \frac{2(x^2+y^2) - 2x(2x)}{(x^2+y^2)^2} = \frac{2x^2 + 2y^2 - 4x^2}{(x^2+y^2)^2} = \frac{2(y^2 - x^2)}{(x^2+y^2)^2}
$$

3. **Compute $u_{yy}$ by Symmetry**:
   Swapping $x$ and $y$:

$$
   u_{yy} = \frac{2(x^2 - y^2)}{(x^2+y^2)^2}
$$

4. **Sum Partials**:

$$
   u_{xx} + u_{yy} = \frac{2(y^2 - x^2) + 2(x^2 - y^2)}{(x^2+y^2)^2} = \frac{0}{(x^2+y^2)^2} = 0
$$

$$
\boxed{\nabla^2 u = u_{xx} + u_{yy} = 0.}
$$

**Key takeaway** — Harmonic functions satisfy $u_{xx} + u_{yy} = 0$. In 2D, logarithmic potential is the fundamental harmonic solution outside the origin.

Recomputing Problem L2.3 from scratch, to check the boxed answer.

In [21]:
u = sp.log(x**2 + y**2)
lap = sp.simplify(sp.diff(u, x, 2) + sp.diff(u, y, 2))
print("u_xx =", sp.simplify(sp.diff(u, x, 2)))
print("u_yy =", sp.simplify(sp.diff(u, y, 2)))
print("Laplacian =", lap)
assert lap == 0
# In three variables ln(x^2+y^2+z^2) is NOT harmonic -- the exponent is dimension-specific.
u3 = sp.log(x**2 + y**2 + z**2)
print("3D log Laplacian =", sp.simplify(sum(sp.diff(u3, v, 2) for v in (x, y, z))))

u_xx = 2*(-x**2 + y**2)/(x**2 + y**2)**2
u_yy = 2*(x**2 - y**2)/(x**2 + y**2)**2
Laplacian = 0
3D log Laplacian = 2/(x**2 + y**2 + z**2)


`u_xx` and `u_yy` are exact negatives of each other. The last line is the contrast: `ln(x^2+y^2+z^2)` has Laplacian `2/r^2` in three variables, so the logarithm is harmonic in the plane only.

### Problem L2.4 — A Maxwell Relation from Schwarz's Theorem
**Source:** Thermodynamics / Callen, *Thermodynamics and an Introduction to Thermostatistics*.

**Statement**
Starting from the first law of thermodynamics for internal energy $dU = T dS - P dV$, where $U(S,V)$ is a smooth state function, derive the Maxwell relation:

$$
\left( \frac{\partial T}{\partial V} \right)_S = -\left( \frac{\partial P}{\partial S} \right)_V
$$

**Intuition**

Because energy $U$ is a state function, $dU = \frac{\partial U}{\partial S} dS + \frac{\partial U}{\partial V} dV$ is an exact total differential. Applying Clairaut's theorem to $U_{SV} = U_{VS}$ yields the Maxwell relation.

**Solution**

1. **Identify Partial Derivatives of $U(S,V)$**:
   Comparing $dU = T dS - P dV$ with $dU = \left(\frac{\partial U}{\partial S}\right)_V dS + \left(\frac{\partial U}{\partial V}\right)_S dV$:

$$
   \left( \frac{\partial U}{\partial S} \right)_V = T, \quad \left( \frac{\partial U}{\partial V} \right)_S = -P
$$

2. **Apply Clairaut's Theorem to Second Mixed Partials**:
   Since $U$ is $C^2$:

$$
   \frac{\partial^2 U}{\partial V \partial S} = \frac{\partial^2 U}{\partial S \partial V}
$$

3. **Substitute First Partials**:

$$
   \frac{\partial}{\partial V}\left[ \left(\frac{\partial U}{\partial S}\right)_V \right]_S = \left(\frac{\partial T}{\partial V}\right)_S
$$

$$
   \frac{\partial}{\partial S}\left[ \left(\frac{\partial U}{\partial V}\right)_S \right]_V = \frac{\partial}{\partial S}[-P]_V = -\left(\frac{\partial P}{\partial S}\right)_V
$$

4. **Equate**:

$$
   \left( \frac{\partial T}{\partial V} \right)_S = -\left( \frac{\partial P}{\partial S} \right)_V
$$

$$
\boxed{\left( \frac{\partial T}{\partial V} \right)_S = -\left( \frac{\partial P}{\partial S} \right)_V.}
$$

**Key takeaway** — Thermodynamic Maxwell relations are direct physical consequences of Clairaut's theorem on mixed partial derivatives of thermodynamic potentials.

Recomputing Problem L2.4 from scratch, to check the boxed answer.

In [22]:
S, V = sp.symbols("S V", positive=True)
U = sp.Function("U")(S, V)
T_, P_ = sp.diff(U, S), -sp.diff(U, V)
residual = sp.simplify(sp.diff(T_, V) + sp.diff(P_, S))
print("(dT/dV)_S + (dP/dS)_V =", residual, " for a general C^2 potential U(S,V)")
assert residual == 0
Uc = S**3 / V
Tc, Pc = sp.diff(Uc, S), -sp.diff(Uc, V)
print("concrete U = S^3/V :  T =", Tc, "  P =", Pc)
print("  (dT/dV)_S =", sp.diff(Tc, V), "   -(dP/dS)_V =", -sp.diff(Pc, S))
assert sp.simplify(sp.diff(Tc, V) + sp.diff(Pc, S)) == 0

(dT/dV)_S + (dP/dS)_V = 0  for a general C^2 potential U(S,V)
concrete U = S^3/V :  T = 3*S**2/V   P = S**3/V**2
  (dT/dV)_S = -3*S**2/V**2    -(dP/dS)_V = -3*S**2/V**2


The identity holds for a completely unspecified `C^2` potential `U(S,V)` — the physics enters only through naming the two first partials `T` and `-P`. The concrete `U = S^3/V` gives `(dT/dV)_S = -(dP/dS)_V = -3S^2/V^2`.

### Problem L2.5 — Gradient of the Least-Squares Loss
**Source:** Machine Learning Foundations / Optimization Theory.

**Statement**
For a linear regression model $\hat{y}_i = w x_i + b$ with Mean Squared Error (MSE) loss function:

$$
L(w, b) = \frac{1}{2N} \sum_{i=1}^N (w x_i + b - y_i)^2
$$

Derive the partial derivatives $\frac{\partial L}{\partial w}$ and $\frac{\partial L}{\partial b}$.

**Intuition**

Gradient descent in ML updates parameters along $-\nabla L = -\left(\frac{\partial L}{\partial w}, \frac{\partial L}{\partial b}\right)^T$. We apply the chain rule inside the sum.

**Solution**

1. **Partial Derivative w.r.t. $w$**:

$$
   \begin{aligned}
   \frac{\partial L}{\partial w} &= \frac{1}{2N} \sum_{i=1}^N \frac{\partial}{\partial w} \left[ (w x_i + b - y_i)^2 \right] \\
   &= \frac{1}{2N} \sum_{i=1}^N 2 (w x_i + b - y_i) \cdot x_i \\
   &= \frac{1}{N} \sum_{i=1}^N (\hat{y}_i - y_i) x_i
   \end{aligned}
$$

2. **Partial Derivative w.r.t. $b$**:

$$
   \begin{aligned}
   \frac{\partial L}{\partial b} &= \frac{1}{2N} \sum_{i=1}^N \frac{\partial}{\partial b} \left[ (w x_i + b - y_i)^2 \right] \\
   &= \frac{1}{2N} \sum_{i=1}^N 2 (w x_i + b - y_i) \cdot 1 \\
   &= \frac{1}{N} \sum_{i=1}^N (\hat{y}_i - y_i)
   \end{aligned}
$$

$$
\boxed{\frac{\partial L}{\partial w} = \frac{1}{N} \sum_{i=1}^N (\hat{y}_i - y_i) x_i, \quad \frac{\partial L}{\partial b} = \frac{1}{N} \sum_{i=1}^N (\hat{y}_i - y_i).}
$$

**Key takeaway** — The weight derivative is the average feature-weighted error, while the bias derivative is the plain average error across samples.

Recomputing Problem L2.5 from scratch, to check the boxed answer.

In [23]:
N = 50
xs = rng.normal(size=N)
ys = 2.5 * xs - 0.7 + 0.1 * rng.normal(size=N)
loss = lambda w, b: 0.5 * np.mean((w * xs + b - ys) ** 2)
w0, b0 = 0.3, -1.1
res = w0 * xs + b0 - ys
gw, gb = np.mean(res * xs), np.mean(res)
h = 1e-6
gw_fd = (loss(w0 + h, b0) - loss(w0 - h, b0)) / (2 * h)
gb_fd = (loss(w0, b0 + h) - loss(w0, b0 - h)) / (2 * h)
print(f"analytic  dL/dw = {gw:+.10f}   dL/db = {gb:+.10f}")
print(f"central   dL/dw = {gw_fd:+.10f}   dL/db = {gb_fd:+.10f}")
print(f"residual  {abs(gw-gw_fd):.2e}   {abs(gb-gb_fd):.2e}")
assert abs(gw - gw_fd) < 1e-6 and abs(gb - gb_fd) < 1e-6
# The stationary point of the gradient is the least-squares fit.
A = np.column_stack([xs, np.ones(N)])
print("normal-equation solution (w, b) =", np.linalg.lstsq(A, ys, rcond=None)[0])

analytic  dL/dw = -2.1368610211   dL/db = -0.4005340025
central   dL/dw = -2.1368610212   dL/db = -0.4005340022
residual  1.35e-10   2.74e-10
normal-equation solution (w, b) = [ 2.500733 -0.683575]


The analytic gradient and the central difference agree to `1e-10`. The stationary point of that gradient is the least-squares line, printed on the last row.

### Problem L2.6 — The Softmax Jacobian
**Source:** Deep Learning Mathematics / Softmax Derivative Derivation.

**Statement**
Given the Softmax function $\sigma: \mathbb{R}^K \to \mathbb{R}^K$ defined by $\sigma(z)_i = \frac{e^{z_i}}{S}$ where $S = \sum_{k=1}^K e^{z_k}$, prove that the Jacobian entries are:

$$
\frac{\partial \sigma_i}{\partial z_j} = \begin{cases} \sigma_i (1 - \sigma_i), & i = j \\ -\sigma_i \sigma_j, & i \neq j \end{cases}
$$

**Intuition**

Softmax normalizes logit inputs into a probability distribution. Changing input $z_j$ affects both the numerator $e^{z_i}$ (if $i=j$) and the denominator $S = \sum e^{z_k}$ (always).

**Solution**

1. **Case $i = j$**:
   Use quotient rule on $\sigma_i = \frac{e^{z_i}}{S}$:

$$
   \frac{\partial \sigma_i}{\partial z_i} = \frac{\frac{\partial}{\partial z_i}(e^{z_i}) \cdot S - e^{z_i} \cdot \frac{\partial S}{\partial z_i}}{S^2}
$$

   Since $\frac{\partial S}{\partial z_i} = e^{z_i}$:

$$
   \frac{\partial \sigma_i}{\partial z_i} = \frac{e^{z_i} S - e^{z_i} e^{z_i}}{S^2} = \frac{e^{z_i}}{S} \left( \frac{S - e^{z_i}}{S} \right) = \sigma_i (1 - \sigma_i)
$$

2. **Case $i \neq j$**:
   Here $\frac{\partial}{\partial z_j}(e^{z_i}) = 0$, but $\frac{\partial S}{\partial z_j} = e^{z_j}$:

$$
   \frac{\partial \sigma_i}{\partial z_j} = \frac{0 \cdot S - e^{z_i} e^{z_j}}{S^2} = -\left(\frac{e^{z_i}}{S}\right) \left(\frac{e^{z_j}}{S}\right) = -\sigma_i \sigma_j
$$

3. **Compact Matrix Representation**:

$$
   J_\sigma(z) = \operatorname{diag}(\sigma(z)) - \sigma(z) \sigma(z)^T
$$

$$
\boxed{\frac{\partial \sigma_i}{\partial z_j} = \sigma_i \delta_{ij} - \sigma_i \sigma_j; \quad J_\sigma(z) = \operatorname{diag}(\sigma) - \sigma \sigma^T.}
$$

**Key takeaway** — Diagonal terms act like logistic variance $\sigma_i(1-\sigma_i)$, while off-diagonal terms $-\sigma_i \sigma_j$ reflect competition between classes.

Recomputing Problem L2.6 from scratch, to check the boxed answer.

In [24]:
from scipy.special import softmax

K = 5
zv = rng.normal(size=K)
s = softmax(zv)
J = np.diag(s) - np.outer(s, s)
h = 1e-6
Jfd = np.column_stack([
    (softmax(zv + h * np.eye(K)[j]) - softmax(zv - h * np.eye(K)[j])) / (2 * h) for j in range(K)
])
print("hand-rolled diag(s) - s s^T:\n", J)
print("max |J - J_finite_difference| =", np.abs(J - Jfd).max())
assert np.abs(J - Jfd).max() < 1e-8
# Rows and columns sum to zero: softmax outputs are constrained to the simplex.
print("row sums:", J.sum(axis=1), "  column sums:", J.sum(axis=0))
assert np.abs(J.sum(axis=1)).max() < 1e-14

hand-rolled diag(s) - s s^T:
 [[ 0.238873 -0.07751  -0.060492 -0.035076 -0.065795]
 [-0.07751   0.157869 -0.030125 -0.017468 -0.032766]
 [-0.060492 -0.030125  0.129822 -0.013633 -0.025572]
 [-0.035076 -0.017468 -0.013633  0.081005 -0.014828]
 [-0.065795 -0.032766 -0.025572 -0.014828  0.13896 ]]
max |J - J_finite_difference| = 3.048647445602626e-11
row sums: [0. 0. 0. 0. 0.]   column sums: [0. 0. 0. 0. 0.]


`diag(s) - s s^T` reproduces the finite-difference Jacobian to `3e-11`. Every row and column sums to zero, because softmax outputs are confined to the probability simplex and no perturbation can change their total.

### Problem L2.7 — Gâteaux Derivative of a Regularised Functional
**Source:** Calculus of Variations / Functional Optimization in AI.

**Statement**
Compute the Gâteaux derivative $d_G J(f; \eta) = \lim_{\epsilon \to 0} \frac{J(f + \epsilon \eta) - J(f)}{\epsilon}$ for the regularized functional:

$$
J(f) = \int_{\Omega} \left( (f(x) - y(x))^2 + \lambda \|\nabla f(x)\|^2 \right) dx
$$

**Intuition**

The Gâteaux derivative of a functional extends directional differentiation to function spaces. We perturb $f$ to $f + \epsilon \eta$ and differentiate w.r.t. scalar $\epsilon$ at $\epsilon = 0$.

**Solution**

1. **Substitute Perturbed Function**:

$$
   J(f + \epsilon \eta) = \int_\Omega \left( (f + \epsilon \eta - y)^2 + \lambda \|\nabla f + \epsilon \nabla \eta\|^2 \right) dx
$$

2. **Differentiate w.r.t. $\epsilon$ inside Integral**:

$$
   \left. \frac{d}{d\epsilon} J(f + \epsilon \eta) \right\vert_{\epsilon=0} = \int_\Omega \left( 2(f - y)\eta + 2\lambda \nabla f \cdot \nabla \eta \right) dx
$$

3. **Apply Integration by Parts (Green's First Identity)**:
   Recall $\int_\Omega \nabla f \cdot \nabla \eta \, dx = -\int_\Omega (\nabla^2 f) \eta \, dx + \int_{\partial \Omega} \eta \frac{\partial f}{\partial n} \, dS$.
   Assuming boundary perturbations vanish ($\eta \big\vert_{\partial\Omega} = 0$):

$$
   d_G J(f; \eta) = 2 \int_\Omega \left( (f(x) - y(x)) - \lambda \nabla^2 f(x) \right) \eta(x) \, dx
$$

$$
\boxed{d_G J(f; \eta) = 2 \int_\Omega \left( (f(x) - y(x)) - \lambda \nabla^2 f(x) \right) \eta(x) \, dx.}
$$

**Key takeaway** — Setting the Gâteaux derivative to zero for all test functions $\eta$ yields the Euler-Lagrange PDE: $(f - y) - \lambda \nabla^2 f = 0$.

Recomputing Problem L2.7 from scratch, to check the boxed answer.

In [25]:
n, lam = 200, 0.3
hgrid = 1.0 / (n + 1)
grid = np.linspace(0.0, 1.0, n + 2)[1:-1]
yv = np.sin(np.pi * grid)
fv = grid * (1.0 - grid)
eta = np.sin(3 * np.pi * grid)          # vanishes on the boundary, as the derivation assumes


def J(fvec):
    full = np.concatenate(([0.0], fvec, [0.0]))
    return np.sum((fvec - yv) ** 2) * hgrid + lam * np.sum((np.diff(full) / hgrid) ** 2) * hgrid


eps = 1e-6
numeric = (J(fv + eps * eta) - J(fv - eps * eta)) / (2 * eps)
full = np.concatenate(([0.0], fv, [0.0]))
lap = (full[:-2] - 2 * full[1:-1] + full[2:]) / hgrid**2
euler = 2 * np.sum(((fv - yv) - lam * lap) * eta) * hgrid
print(f"difference quotient of J   {numeric:+.10f}")
print(f"Euler-Lagrange integral    {euler:+.10f}")
print(f"residual                   {abs(numeric-euler):.3e}")
assert abs(numeric - euler) < 1e-6

difference quotient of J   +0.2641572611
Euler-Lagrange integral    +0.2641572611
residual                   2.033e-11


The discretised difference quotient of `J` and the discretised Euler-Lagrange integral agree to `2e-11`. The summation by parts on the grid is the discrete image of Green's first identity, and the boundary terms drop because `eta` vanishes at both ends.

### Problem L2.8 — Error Propagation in Dissipated Power
**Source:** Experimental Physics / Error Analysis.

**Statement**
Electrical power dissipated in a resistor is $P = \frac{V^2}{R}$. If voltage $V$ has a relative measurement uncertainty of $\left\vert\frac{\Delta V}{V}\right\vert = 1\%$ and resistance $R$ has an uncertainty of $\left\vert\frac{\Delta R}{R}\right\vert = 2\%$, use total differentials to estimate the maximum relative uncertainty $\left\vert\frac{\Delta P}{P}\right\vert$.

**Intuition**

The total differential $dP = P_V dV + P_R dR$ models error propagation. Dividing by $P$ converts absolute differentials into relative percentage errors.

**Solution**

1. **Compute Partials of $P(V,R) = V^2 R^{-1}$**:

$$
   P_V = \frac{2V}{R}, \quad P_R = -\frac{V^2}{R^2}
$$

2. **Formulate Differential $dP$**:

$$
   dP = \frac{2V}{R} dV - \frac{V^2}{R^2} dR
$$

3. **Divide by $P = \frac{V^2}{R}$**:

$$
   \frac{dP}{P} = \frac{\frac{2V}{R} dV}{\frac{V^2}{R}} - \frac{\frac{V^2}{R^2} dR}{\frac{V^2}{R}} = 2 \frac{dV}{V} - \frac{dR}{R}
$$

4. **Bound the Linear Model via the Triangle Inequality**:

$$
   \left\vert \frac{dP}{P} \right\vert \le 2 \left\vert \frac{dV}{V} \right\vert + \left\vert \frac{dR}{R} \right\vert = 2(1\%) + 2\% = 4\%
$$

5. **Read the Bound Correctly**:
   This bounds the *differential* $dP/P$, which is the first-order model of $\Delta P/P$, not
   $\Delta P/P$ itself. The true worst case is attained at $dV/V = +1\%$, $dR/R = -2\%$:

$$
   \frac{\Delta P}{P} = \frac{(1.01)^2}{0.98} - 1 = 0.040918\ldots = 4.0918\%
$$

   which exceeds $4\%$ by the second-order term. The code cell below evaluates all four sign
   combinations exactly.

$$
\boxed{\left\vert \frac{dP}{P} \right\vert \le 2\left\vert \frac{dV}{V}\right\vert + \left\vert \frac{dR}{R}\right\vert = 4\% \;\text{(first order)}; \quad \text{exact worst case } 4.0918\%.}
$$

**Key takeaway** — Exponents in a power law become multiplier weights in relative error propagation ($V^2 \to 2 \cdot 1\%$), and the resulting bound is first order: the exact excursion may exceed it.

Recomputing Problem L2.8 from scratch, to check the boxed answer.

In [26]:
V0, R0 = 12.0, 100.0
P = lambda Vv, Rr: Vv**2 / Rr
worst = 0.0
for sv in (+1, -1):
    for sr in (+1, -1):
        rel = (P(V0 * (1 + sv * 0.01), R0 * (1 + sr * 0.02)) - P(V0, R0)) / P(V0, R0)
        worst = max(worst, abs(rel))
        print(f"dV/V = {sv*1:+d}%   dR/R = {sr*2:+d}%   exact dP/P = {rel*100:+.4f}%   linear = {(2*sv*1 - sr*2):+d}%")
print(f"worst exact excursion {worst*100:.4f}%   first-order bound 4.0000%")
assert abs(worst - 0.04) < 2e-3

dV/V = +1%   dR/R = +2%   exact dP/P = +0.0098%   linear = +0%
dV/V = +1%   dR/R = -2%   exact dP/P = +4.0918%   linear = +4%
dV/V = -1%   dR/R = +2%   exact dP/P = -3.9118%   linear = -4%
dV/V = -1%   dR/R = -2%   exact dP/P = +0.0102%   linear = +0%
worst exact excursion 4.0918%   first-order bound 4.0000%


The first-order bound `4%` is the linear model, not an exact envelope: the true worst case here is `4.0918%`, which exceeds it because `1/(1 - 0.02)` is slightly larger than `1 + 0.02`. Total differentials bound the error to first order only.

### Problem L2.9 — Unit Normal to an Equipotential Surface
**Source:** Classical Electrodynamics (Jackson / Marsden & Tromba).

**Statement**
Find a unit normal vector $n$ to the equipotential surface $V(x,y,z) = \frac{1}{\sqrt{x^2+y^2+z^2}} = \frac{1}{3}$ at point $P(1, 2, 2)$.

**Intuition**

The gradient $\nabla V(P)$ is always orthogonal (normal) to the level surface $V(x,y,z) = C$ passing through $P$. Normalizing $\nabla V(P)$ yields the unit normal vector.

**Solution**

1. **Verify Point $P(1,2,2)$**:

$$
   V(1,2,2) = \frac{1}{\sqrt{1^2+2^2+2^2}} = \frac{1}{\sqrt{9}} = \frac{1}{3} \quad \checkmark
$$

2. **Compute Gradient $\nabla V$**:
   Let $r = \sqrt{x^2+y^2+z^2}$, so $V = r^{-1}$.

$$
   \nabla V = -\frac{1}{r^2} \nabla r = -\frac{1}{r^3} (x, y, z)
$$

3. **Evaluate Gradient at $P(1,2,2)$ where $r = 3$**:

$$
   \nabla V(1,2,2) = -\frac{1}{27} (1, 2, 2)
$$

4. **Normalize to Unit Vector**:

$$
   n = \frac{\nabla V(1,2,2)}{\|\nabla V(1,2,2)\|} = -\frac{(1, 2, 2)}{\sqrt{1^2+2^2+2^2}} = -\frac{1}{3} (1, 2, 2)
$$

   (The outward unit normal is $+\frac{1}{3}(1,2,2)$ or inward $-\frac{1}{3}(1,2,2)$).

$$
\boxed{n = \pm \left( \frac{1}{3}, \frac{2}{3}, \frac{2}{3} \right).}
$$

**Key takeaway** — Gradients are normal to level sets: $n = \pm \frac{\nabla V}{\|\nabla V\|}$.

Recomputing Problem L2.9 from scratch, to check the boxed answer.

In [27]:
V = 1 / sp.sqrt(x**2 + y**2 + z**2)
grad = sp.Matrix([sp.diff(V, v) for v in (x, y, z)])
g0 = sp.simplify(grad.subs({x: 1, y: 2, z: 2}))
n = sp.simplify(g0 / g0.norm())
print("V(1,2,2) =", sp.simplify(V.subs({x: 1, y: 2, z: 2})))
print("grad V(1,2,2) =", g0.T, "   ||grad V|| =", sp.simplify(g0.norm()))
print("unit normal =", n.T)
assert sp.simplify(n - sp.Matrix([-sp.Rational(1, 3), -sp.Rational(2, 3), -sp.Rational(2, 3)])) == sp.zeros(3, 1)
# The normal is orthogonal to every tangent direction of the level surface.
tangent = sp.Matrix([2, -1, 0])          # satisfies tangent . (1,2,2) = 0
print("n . tangent =", sp.simplify((n.T * tangent)[0]))

V(1,2,2) = 1/3
grad V(1,2,2) = Matrix([[-1/27, -2/27, -2/27]])    ||grad V|| = 1/9
unit normal = Matrix([[-1/3, -2/3, -2/3]])
n . tangent = 0


`grad V(1,2,2) = -(1,2,2)/27` and its norm is `1/9`, giving the unit normal `-(1,2,2)/3`. It is orthogonal to `(2,-1,0)`, a tangent direction of the sphere at that point, so the gradient really is normal to the level surface.

### Problem L2.10 — Forward-Mode Automatic Differentiation
**Source:** Computational Mathematics / Automatic Differentiation Dual Numbers.

**Statement**
Using forward-mode automatic differentiation with dual numbers $a + b \epsilon$ (where $\epsilon^2 = 0$), compute the value and partial derivative $\frac{\partial f}{\partial x}$ of $f(x,y) = x^2 y + \sin(x)$ at $(x,y) = (2, 3)$.

**Intuition**

Dual numbers track function values in the real part and directional derivative seeds in the $\epsilon$ component: $f(x + \epsilon, y) = f(x,y) + \frac{\partial f}{\partial x} \epsilon$.

**Solution**

1. **Set Input Dual Variables for $\frac{\partial}{\partial x}$ at $(2, 3)$**:

$$
   X = 2 + 1 \cdot \epsilon, \quad Y = 3 + 0 \cdot \epsilon
$$

2. **Compute $X^2$**:

$$
   X^2 = (2 + \epsilon)^2 = 4 + 4 \epsilon + \epsilon^2 = 4 + 4 \epsilon
$$

3. **Compute $X^2 Y$**:

$$
   X^2 Y = (4 + 4 \epsilon)(3) = 12 + 12 \epsilon
$$

4. **Compute $\sin(X)$ via Taylor Expansion**:

$$
   \sin(2 + \epsilon) = \sin(2) + \cos(2) \epsilon
$$

5. **Sum Terms**:

$$
   f(X, Y) = (12 + 12 \epsilon) + (\sin(2) + \cos(2) \epsilon) = (12 + \sin(2)) + (12 + \cos(2)) \epsilon
$$

6. **Extract Results**:
   - Function value: $f(2,3) = 12 + \sin(2) \approx 12.9093$.
   - Partial derivative $\frac{\partial f}{\partial x}(2,3) = 12 + \cos(2) \approx 11.5839$.

$$
\boxed{f(2,3) = 12 + \sin(2); \quad \frac{\partial f}{\partial x}(2,3) = 12 + \cos(2).}
$$

**Key takeaway** — Forward-mode AD evaluates exact analytical partial derivatives alongside function values without symbolic differentiation or numerical finite-difference errors.

Recomputing Problem L2.10 from scratch, to check the boxed answer.

In [28]:
class Dual:
    def __init__(self, a, b=0.0):
        self.a, self.b = float(a), float(b)

    def __add__(self, o):
        o = o if isinstance(o, Dual) else Dual(o)
        return Dual(self.a + o.a, self.b + o.b)

    def __mul__(self, o):
        o = o if isinstance(o, Dual) else Dual(o)
        return Dual(self.a * o.a, self.a * o.b + self.b * o.a)

    __radd__, __rmul__ = __add__, __mul__

    def __repr__(self):
        return f"{self.a:.10f} + {self.b:.10f}*eps"


dsin = lambda u: Dual(np.sin(u.a), np.cos(u.a) * u.b)
Xd, Yd = Dual(2.0, 1.0), Dual(3.0, 0.0)     # seed epsilon on x
out = Xd * Xd * Yd + dsin(Xd)
print("dual-number result :", out)
print(f"closed form        : {12+np.sin(2):.10f} + {12+np.cos(2):.10f}*eps")
assert abs(out.a - (12 + np.sin(2))) < 1e-12 and abs(out.b - (12 + np.cos(2))) < 1e-12
# Seeding epsilon on y instead returns df/dy = x^2 = 4.
Xd, Yd = Dual(2.0, 0.0), Dual(3.0, 1.0)
print("seed on y :", Xd * Xd * Yd + dsin(Xd), "  -> df/dy = 4")

dual-number result : 12.9092974268 + 11.5838531635*eps
closed form        : 12.9092974268 + 11.5838531635*eps
seed on y : 12.9092974268 + 4.0000000000*eps   -> df/dy = 4


The dual-number arithmetic reproduces `12 + sin(2)` and `12 + cos(2)` to `1e-12` — these are exact derivative values, not finite differences. Moving the seed to `y` returns `df/dy = x^2 = 4`.

### Problem L2.11 — Backpropagation Through Two Layers
**Source:** Deep Learning Theory / Backpropagation Chain Rule.

**Statement**
Consider a 2-layer neural network model $y = W_2 \sigma(W_1 x + b_1) + b_2$ with $x \in \mathbb{R}^{d_0}$, $W_1 \in \mathbb{R}^{d_1 \times d_0}$, $b_1 \in \mathbb{R}^{d_1}$, $W_2 \in \mathbb{R}^{d_2 \times d_1}$, $b_2 \in \mathbb{R}^{d_2}$, and element-wise activation $\sigma$. Derive the Jacobian matrix $\frac{\partial y}{\partial x} \in \mathbb{R}^{d_2 \times d_0}$ using the multivariable chain rule.

**Intuition**

The multivariable chain rule states that the Jacobian of a composition of functions is the product of their individual Jacobian matrices: $J_{f \circ g}(x) = J_f(g(x)) J_g(x)$.

**Solution**

1. **Break Down into Intermediate Layers**:
   - $z_1 = W_1 x + b_1 \in \mathbb{R}^{d_1}$
   - $h_1 = \sigma(z_1) \in \mathbb{R}^{d_1}$
   - $y = W_2 h_1 + b_2 \in \mathbb{R}^{d_2}$

2. **Compute Layer Jacobians**:
   - $\frac{\partial z_1}{\partial x} = W_1 \in \mathbb{R}^{d_1 \times d_0}$
   - $\frac{\partial h_1}{\partial z_1} = \operatorname{diag}(\sigma'(z_1)) \in \mathbb{R}^{d_1 \times d_1}$
   - $\frac{\partial y}{\partial h_1} = W_2 \in \mathbb{R}^{d_2 \times d_1}$

3. **Apply Jacobian Chain Rule**:

$$
   \frac{\partial y}{\partial x} = \frac{\partial y}{\partial h_1} \cdot \frac{\partial h_1}{\partial z_1} \cdot \frac{\partial z_1}{\partial x}
$$

   Substitute:

$$
   \frac{\partial y}{\partial x} = W_2 \cdot \operatorname{diag}(\sigma'(z_1)) \cdot W_1
$$

$$
\boxed{\frac{\partial y}{\partial x} = W_2 \, \operatorname{diag}(\sigma'(W_1 x + b_1)) \, W_1.}
$$

**Key takeaway** — Deep network differentiation is exact matrix multiplication of layer Jacobians via the chain rule.

Recomputing Problem L2.11 from scratch, to check the boxed answer.

In [29]:
d0, d1, d2 = 4, 6, 3
W1, b1 = rng.normal(size=(d1, d0)), rng.normal(size=d1)
W2, b2 = rng.normal(size=(d2, d1)), rng.normal(size=d2)
sig, dsig = np.tanh, lambda u: 1.0 - np.tanh(u) ** 2
net = lambda v: W2 @ sig(W1 @ v + b1) + b2
xv = rng.normal(size=d0)
J = W2 @ np.diag(dsig(W1 @ xv + b1)) @ W1
h = 1e-6
Jfd = np.column_stack([(net(xv + h * np.eye(d0)[j]) - net(xv - h * np.eye(d0)[j])) / (2 * h) for j in range(d0)])
print("chain-rule Jacobian shape", J.shape, " expected", (d2, d0))
print("max |J - J_finite_difference| =", np.abs(J - Jfd).max())
assert J.shape == (d2, d0) and np.abs(J - Jfd).max() < 1e-7
# Reversing the product order is not merely wrong numerically -- the shapes do not conform.
try:
    _ = W1 @ np.diag(dsig(W1 @ xv + b1)) @ W2
except ValueError as err:
    print("W1 diag(sigma') W2 fails:", err)

chain-rule Jacobian shape (3, 4)  expected (3, 4)
max |J - J_finite_difference| = 5.028168853504766e-10
W1 diag(sigma') W2 fails: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 6 is different from 4)


The hand-rolled product `W2 diag(sigma') W1` matches the finite-difference Jacobian to `3e-10` and has the shape `(d2, d0)`. Reversing the factors does not merely give the wrong number: the shapes do not conform, which is the cheapest guard against writing the chain rule backwards.

### Problem L2.12 — Incompressible Flow Is Divergence-Free
**Source:** Fluid Mechanics / Incompressible Flow Theory.

**Statement**
The conservation of mass for a fluid with density $\rho(x,y,z,t)$ and velocity field $\mathbf{u} = (u_x, u_y, u_z)$ is governed by the continuity equation:

$$
\frac{\partial \rho}{\partial t} + \nabla \cdot (\rho \mathbf{u}) = 0
$$

Show that for an incompressible fluid ($\rho = \text{constant}$), mass conservation reduces to the vanishing divergence condition $\frac{\partial u_x}{\partial x} + \frac{\partial u_y}{\partial y} + \frac{\partial u_z}{\partial z} = 0$.

**Intuition**

Incompressibility means fluid density does not change along fluid element trajectories. Taking constant $\rho$ out of partial derivatives simplifies the continuity equation.

**Solution**

1. **Expand $\nabla \cdot (\rho \mathbf{u})$ using Product Rule for Partials**:

$$
   \nabla \cdot (\rho \mathbf{u}) = \frac{\partial(\rho u_x)}{\partial x} + \frac{\partial(\rho u_y)}{\partial y} + \frac{\partial(\rho u_z)}{\partial z}
$$

   Apply product rule to each term:

$$
   \frac{\partial(\rho u_x)}{\partial x} = u_x \frac{\partial \rho}{\partial x} + \rho \frac{\partial u_x}{\partial x}
$$

   Summing yields:

$$
   \nabla \cdot (\rho \mathbf{u}) = \mathbf{u} \cdot \nabla \rho + \rho (\nabla \cdot \mathbf{u})
$$

2. **Substitute into Continuity Equation**:

$$
   \frac{\partial \rho}{\partial t} + \mathbf{u} \cdot \nabla \rho + \rho (\nabla \cdot \mathbf{u}) = 0
$$

3. **Apply Incompressibility Condition**:
   If $\rho$ is constant in space and time, $\frac{\partial \rho}{\partial t} = 0$ and $\nabla \rho = \mathbf{0}$.
   The equation reduces to:

$$
   \rho (\nabla \cdot \mathbf{u}) = 0 \implies \nabla \cdot \mathbf{u} = 0
$$

   In partial derivative components:

$$
   \frac{\partial u_x}{\partial x} + \frac{\partial u_y}{\partial y} + \frac{\partial u_z}{\partial z} = 0
$$

$$
\boxed{\frac{\partial u_x}{\partial x} + \frac{\partial u_y}{\partial y} + \frac{\partial u_z}{\partial z} = 0.}
$$

**Key takeaway** — Mass conservation for constant-density fluids requires the velocity field to be solenoidal (divergence-free).

Recomputing Problem L2.12 from scratch, to check the boxed answer.

In [30]:
rho = sp.Function("rho")(x, y, z)
ux, uy, uz = (sp.Function(nm)(x, y, z) for nm in ("u_x", "u_y", "u_z"))
lhs = sp.diff(rho * ux, x) + sp.diff(rho * uy, y) + sp.diff(rho * uz, z)
rhs = (ux * sp.diff(rho, x) + uy * sp.diff(rho, y) + uz * sp.diff(rho, z)
       + rho * (sp.diff(ux, x) + sp.diff(uy, y) + sp.diff(uz, z)))
print("div(rho u) - [u.grad rho + rho div u] =", sp.simplify(lhs - rhs))
assert sp.simplify(lhs - rhs) == 0
u = (sp.sin(x) * sp.cos(y), -sp.cos(x) * sp.sin(y), sp.Integer(0))
div_u = sp.simplify(sp.diff(u[0], x) + sp.diff(u[1], y) + sp.diff(u[2], z))
print("concrete field (sin x cos y, -cos x sin y, 0):  div u =", div_u)
assert div_u == 0
bad = (x, y, z)
print("radial field (x, y, z):  div u =", sp.diff(bad[0], x) + sp.diff(bad[1], y) + sp.diff(bad[2], z), "-> compressible")

div(rho u) - [u.grad rho + rho div u] = 0
concrete field (sin x cos y, -cos x sin y, 0):  div u = 0
radial field (x, y, z):  div u = 3 -> compressible


The product rule for the divergence holds symbolically for arbitrary `rho` and `u`. The concrete field `(sin x cos y, -cos x sin y, 0)` is solenoidal, while the radial field `(x,y,z)` has divergence `3` and therefore cannot describe an incompressible flow.

## L3 — Challenge Proofs

### Problem L3.1 — Peano's Function: Mixed Partials Disagree
**Source:** Demidovich, *Problems in Mathematical Analysis*, No. 3217 (Peano's Counterexample).

**Statement**
Rigorously prove that Peano's function:

$$
f(x,y) = \begin{cases} \frac{x y (x^2 - y^2)}{x^2 + y^2}, & (x,y) \neq (0,0) \\ 0, & (x,y) = (0,0) \end{cases}
$$

satisfies $f_{xy}(0,0) = -1$ and $f_{yx}(0,0) = +1$, proving non-equality of mixed partials at $(0,0)$.

**Intuition**

We must compute the mixed partials at $(0,0)$ using the definition of partial derivatives applied to $f_x(0,y)$ and $f_y(x,0)$.

**Solution**

*(Example 6.4 of `first_principles.ipynb` derives the partial derivatives away from the origin. Here we work the limits at the origin itself.)*

1. **Evaluate First Partials on the Axes**:
   From the definition or direct computation (as shown in the curriculum), we find that along the axes:

$$
   f_x(0, y) = -y \quad \text{and} \quad f_y(x, 0) = x
$$

   Additionally, $f_x(0,0) = 0$ and $f_y(0,0) = 0$.

2. **Compute Mixed Partials at the Origin**:
   Using the limit definition of partial derivatives:

$$
   f_{xy}(0,0) = \lim_{k \to 0} \frac{f_x(0, k) - f_x(0,0)}{k} = \lim_{k \to 0} \frac{-k - 0}{k} = -1
$$

$$
   f_{yx}(0,0) = \lim_{h \to 0} \frac{f_y(h, 0) - f_y(0,0)}{h} = \lim_{h \to 0} \frac{h - 0}{h} = +1
$$

3. **Conclusion**:
   Since $-1 \neq +1$, we have $f_{xy}(0,0) \neq f_{yx}(0,0)$.

4. **Locate the discontinuity of $f_{xy}$.**
   Away from the origin the quotient rule applied to $f_x$ gives

$$
   f_{xy}(x,y) = \frac{x^6 + 9 x^4 y^2 - 9 x^2 y^4 - y^6}{(x^2+y^2)^3}, \qquad (x,y) \neq (0,0)
$$

   Along the diagonal $y = x$ the numerator is $x^6 + 9x^6 - 9x^6 - x^6 = 0$, so

$$
   f_{xy}(x,x) = 0 \quad \text{for every } x \neq 0
$$

   while along the $y$-axis $f_{xy}(0,y) = -y^6/y^6 = -1$. Two paths, two different values, so
   $f_{xy}$ has no limit at the origin and is discontinuous there. That is exactly the hypothesis
   Theorem 4.4 requires, and exactly what this $f$ lacks.

$$
\boxed{f_{xy}(0,0) = -1, \quad f_{yx}(0,0) = +1; \quad f_{xy} \text{ is discontinuous at } (0,0).}
$$

**Key takeaway** — Mixed partial derivatives fail to commute when second partials are discontinuous at the evaluation point.

Recomputing Problem L3.1 from scratch, to check the boxed answer.

In [31]:
h, k = sp.symbols("h k", positive=True)
X, Y = sp.symbols("X Y", real=True)
F = X * Y * (X**2 - Y**2) / (X**2 + Y**2)          # f(0,Y) = f(X,0) = 0
fx_0Y = sp.simplify(sp.limit(F.subs(X, h) / h, h, 0))
fy_X0 = sp.simplify(sp.limit(F.subs(Y, k) / k, k, 0))
print("f_x(0,Y) =", fx_0Y, "     f_y(X,0) =", fy_X0)
fxy00 = sp.limit(fx_0Y.subs(Y, k) / k, k, 0)
fyx00 = sp.limit(fy_X0.subs(X, h) / h, h, 0)
print("f_xy(0,0) =", fxy00, "   f_yx(0,0) =", fyx00)
assert (fxy00, fyx00) == (-1, 1)
fxy = sp.simplify(sp.diff(F, X, Y))
print("f_xy off the origin on x = 0 :", sp.simplify(fxy.subs(X, 0)))
print("f_xy off the origin on y = x :", sp.simplify(fxy.subs(Y, X)))
assert sp.simplify(fxy.subs(Y, X)) == 0

f_x(0,Y) = -Y      f_y(X,0) = X
f_xy(0,0) = -1    f_yx(0,0) = 1


f_xy off the origin on x = 0 : -1
f_xy off the origin on y = x : 0


SymPy confirms `f_x(0,y) = -y` and `f_y(x,0) = x`, hence `f_xy(0,0) = -1` and `f_yx(0,0) = +1`. Off the origin, `f_xy` equals `-1` along the `y`-axis but `0` along the diagonal `y = x`, so it has no limit at the origin — precisely the continuity hypothesis Theorem 4.4 requires.

### Problem L3.2 — Every Direction, No Continuity
**Source:** Spivak, *Calculus on Manifolds*, Ch. 2.

**Statement**
Let $f(x,y) = \frac{x^2 y}{x^4 + y^2}$ for $(x,y) \neq (0,0)$ and $f(0,0) = 0$. Prove that the directional derivative $D_v f(0,0)$ exists for **every** directional vector $v = (v_1, v_2) \in \mathbb{R}^2$, yet $f$ is not continuous at $(0,0)$.

**Intuition**

This pathological counterexample shows that Gâteaux differentiability in all directions does not imply continuity in $\mathbb{R}^n$.

**Solution**

1. **Compute Directional Derivative $D_v f(0,0)$ for $v = (v_1, v_2)$**:
   By definition:

$$
   D_v f(0,0) = \lim_{t \to 0} \frac{f(0 + t v_1, 0 + t v_2) - f(0,0)}{t} = \lim_{t \to 0} \frac{1}{t} \left( \frac{t^2 v_1^2 \cdot t v_2}{t^4 v_1^4 + t^2 v_2^2} \right)
$$

2. **Case A: $v_2 \neq 0$**:

$$
   D_v f(0,0) = \lim_{t \to 0} \frac{t^3 v_1^2 v_2}{t \cdot t^2 (t^2 v_1^4 + v_2^2)} = \lim_{t \to 0} \frac{v_1^2 v_2}{t^2 v_1^4 + v_2^2} = \frac{v_1^2 v_2}{0 + v_2^2} = \frac{v_1^2}{v_2}
$$

3. **Case B: $v_2 = 0$ ($v = (v_1, 0)$)**:

$$
   f(t v_1, 0) = 0 \implies D_v f(0,0) = 0
$$

   Thus $D_v f(0,0)$ exists for all directions $v \in \mathbb{R}^2$.

4. **Show Discontinuity at $(0,0)$**:
   Approach $(0,0)$ along the parabolic trajectory $y = x^2$:

$$
   f(x, x^2) = \frac{x^2 (x^2)}{x^4 + (x^2)^2} = \frac{x^4}{2 x^4} = \frac{1}{2} \neq f(0,0) = 0
$$

   Thus $f$ is discontinuous at $(0,0)$.

5. **Is $v \mapsto D_v f(0,0)$ linear?** From step 2,
   $D_v f(0,0) = v_1^2/v_2$ for $v_2 \neq 0$, which is quadratic in $v_1$, not linear in $v$.
   So this $f$ fails to be Gâteaux differentiable under the module's standard (linear) definition
   as well — it only satisfies the weaker "all directional derivatives exist" condition (P) of
   Proposition 4.6, which is consistent with $f$ being discontinuous at $(0,0)$ (Example 6.6 of `first_principles.ipynb`).

$$
\boxed{D_v f(0,0) = \begin{cases} \frac{v_1^2}{v_2}, & v_2 \neq 0 \\ 0, & v_2 = 0 \end{cases}; \quad \text{Discontinuous along } y=x^2; \; v \mapsto D_v f(0,0) \text{ not linear.}}
$$

**Key takeaway** — Gâteaux differentiability in all directions tests only linear approaches $t v$ and fails to detect non-linear parabolic discontinuities.

Recomputing Problem L3.2 from scratch, to check the boxed answer.

In [32]:
f = lambda X, Y: X**2 * Y / (X**4 + Y**2)
ts = np.array([1e-4, 1e-6, 1e-8])
for v1, v2 in ((1.0, 1.0), (2.0, 0.5), (1.0, 0.0), (3.0, -2.0)):
    q = f(v1 * ts, v2 * ts) / ts
    pred = v1**2 / v2 if v2 != 0 else 0.0
    print(f"v = ({v1:+.1f},{v2:+.1f})   quotients {q}   predicted {pred:+.4f}")
    assert np.allclose(q, pred, atol=1e-6)
xs = np.array([1e-1, 1e-2, 1e-3, 1e-4])
print("f(x, x^2) =", f(xs, xs**2), "  -> 1/2, never f(0,0) = 0")
assert np.allclose(f(xs, xs**2), 0.5)
# The direction map is quadratic in v1, so it is not linear -- and f is not even continuous.
print("D_(1,1) + D_(0,1) =", 1.0 + 0.0, "   but D_(1,2) =", 1.0**2 / 2.0)

v = (+1.0,+1.0)   quotients [1. 1. 1.]   predicted +1.0000
v = (+2.0,+0.5)   quotients [7.999995 8.       8.      ]   predicted +8.0000
v = (+1.0,+0.0)   quotients [0. 0. 0.]   predicted +0.0000
v = (+3.0,-2.0)   quotients [-4.499999 -4.5      -4.5     ]   predicted -4.5000
f(x, x^2) = [0.5 0.5 0.5 0.5]   -> 1/2, never f(0,0) = 0
D_(1,1) + D_(0,1) = 1.0    but D_(1,2) = 0.5


Every direction gives a finite quotient matching `v_1^2 / v_2`, so all directional derivatives exist. Yet `f` is pinned at `1/2` along `y = x^2`, so `f` is not continuous, and the direction map is quadratic in `v_1`, so it is not linear either.

### Problem L3.3 — Continuous, Directional, Not Differentiable
**Source:** Kaczor & Nowak, *Problems in Real Analysis*, Vol. II.

**Statement**
For $f(x,y) = \frac{x^3}{x^2+y^2}$ for $(x,y) \neq (0,0)$ and $f(0,0)=0$:
1. Prove $f$ is continuous at $(0,0)$.
2. Compute $D_v f(0,0)$ for all $v = (v_1, v_2) \in \mathbb{R}^2$.
3. Prove that $v \mapsto D_v f(0,0)$ is NOT a linear map, and conclude $f$ is NOT Fréchet differentiable at $(0,0)$.

**Intuition**

A necessary condition for Fréchet differentiability is that $D_v f(a) = L(v)$ must be linear in $v$. If $D_v f(a)$ is non-linear in $v$, Fréchet differentiability fails.

**Solution**

1. **Prove Continuity via Polar Coordinates**:
   Let $x = r \cos\theta, y = r \sin\theta$.

$$
   \lvert f(r\cos\theta, r\sin\theta) \rvert = \left\vert \frac{r^3 \cos^3\theta}{r^2} \right\vert = r \lvert\cos^3\theta\rvert \le r \xrightarrow{r \to 0^+} 0 = f(0,0)
$$

   Thus $f$ is continuous at $(0,0)$.

2. **Compute $D_v f(0,0)$**:

$$
   D_v f(0,0) = \lim_{t \to 0} \frac{f(t v_1, t v_2) - 0}{t} = \lim_{t \to 0} \frac{1}{t} \left( \frac{t^3 v_1^3}{t^2 v_1^2 + t^2 v_2^2} \right) = \frac{v_1^3}{v_1^2 + v_2^2}
$$

3. **Check Linearity of Directional Derivative Map**:
   Let $u = (1, 0)$ and $v = (0, 1)$.
   - $D_u f(0,0) = \frac{1^3}{1^2+0^2} = 1$
   - $D_v f(0,0) = \frac{0^3}{0^2+1^2} = 0$
   - Sum of images: $D_u f(0,0) + D_v f(0,0) = 1 + 0 = 1$.

   Now evaluate on $u + v = (1, 1)$:
   - $D_{u+v} f(0,0) = \frac{1^3}{1^2+1^2} = \frac{1}{2}$.

   Since $D_{u+v} f(0,0) = \frac{1}{2} \neq 1 = D_u f(0,0) + D_v f(0,0)$, the map $v \mapsto D_v f(0,0)$ is **non-linear**.
   Therefore, $f$ is **not Fréchet differentiable** at $(0,0)$.

4. **Which hypothesis of Theorem 4.2 fails?** The partials
   $f_x(0,0)=1$, $f_y(0,0)=0$ exist, so this is not a failure of existence. Compute
   $f_x(x,y) = \dfrac{x^2(x^2+3y^2)}{(x^2+y^2)^2}$ for $(x,y)\neq(0,0)$ and let $y=x\to0$:
   $f_x(x,x) = \dfrac{x^2\cdot4x^2}{4x^4} = 1$ for all $x\neq0$, while along $x=0$, $f_x(0,y)=0$.
   So $f_x$ has no limit at the origin — it is discontinuous there, which is exactly the
   hypothesis Theorem 4.2 needs and this example lacks.

$$
\boxed{D_v f(0,0) = \frac{v_1^3}{v_1^2+v_2^2}; \text{ Non-linear in } v \implies \text{Not Fréchet differentiable}; \; f_x \text{ discontinuous at } (0,0).}
$$

**Key takeaway** — Fréchet differentiability requires the directional derivative map to be strictly additive and linear in direction $v$.

Recomputing Problem L3.3 from scratch, to check the boxed answer.

In [33]:
f = lambda X, Y: X**3 / (X**2 + Y**2)
ts = np.array([1e-4, 1e-6, 1e-8])
for v1, v2 in ((1.0, 0.0), (0.0, 1.0), (1.0, 1.0), (2.0, -3.0)):
    q = f(v1 * ts, v2 * ts) / ts
    pred = v1**3 / (v1**2 + v2**2)
    print(f"v = ({v1:+.1f},{v2:+.1f})   quotients {q}   v1^3/(v1^2+v2^2) = {pred:+.6f}")
    assert np.allclose(q, pred, atol=1e-12)
Du, Dv, Duv = 1.0, 0.0, 1.0**3 / (1.0**2 + 1.0**2)
print(f"D_u + D_v = {Du+Dv:.4f}   but D_(u+v) = {Duv:.4f}  -> the direction map is not additive")
assert abs(Duv - (Du + Dv)) > 0.4
# f is nevertheless continuous at the origin: |f| <= r.
r = np.array([1e-2, 1e-4, 1e-6])
th = rng.uniform(0.0, 2 * np.pi, (r.size, 4000))
print("max |f| per radius:", np.abs(f(r[:, None] * np.cos(th), r[:, None] * np.sin(th))).max(axis=1))
# ... and the hypothesis of Theorem 4.2 fails: f_x has no limit at the origin.
fx = lambda X, Y: X**2 * (X**2 + 3 * Y**2) / (X**2 + Y**2) ** 2
print("f_x along y = x :", fx(ts, ts), "   f_x along x = 0 :", fx(0 * ts, ts))

v = (+1.0,+0.0)   quotients [1. 1. 1.]   v1^3/(v1^2+v2^2) = +1.000000
v = (+0.0,+1.0)   quotients [0. 0. 0.]   v1^3/(v1^2+v2^2) = +0.000000
v = (+1.0,+1.0)   quotients [0.5 0.5 0.5]   v1^3/(v1^2+v2^2) = +0.500000
v = (+2.0,-3.0)   quotients [0.615385 0.615385 0.615385]   v1^3/(v1^2+v2^2) = +0.615385
D_u + D_v = 1.0000   but D_(u+v) = 0.5000  -> the direction map is not additive
max |f| per radius: [0.01     0.0001   0.000001]
f_x along y = x : [1. 1. 1.]    f_x along x = 0 : [0. 0. 0.]


The quotients match `v_1^3 / (v_1^2 + v_2^2)` to machine precision for every direction, and `f` is continuous — `|f| <= r`. But `D_u + D_v = 1` while `D_(u+v) = 1/2`, so the map is not additive. The final rows show why: `f_x` equals `1` along `y = x` and `0` along `x = 0`, so it is discontinuous at the origin and Theorem 4.2 does not apply.

### Problem L3.4 — Oscillating Powers and the Fréchet Threshold
**Source:** Adapted from Demidovich / Cambridge Tripos Part IA.

**Statement**
Determine for which values of $\alpha \gt 0$ the function:

$$
f(x,y) = \begin{cases} (x^2+y^2)^\alpha \sin\left(\frac{1}{x^2+y^2}\right), & (x,y) \neq (0,0) \\ 0, & (x,y) = (0,0) \end{cases}
$$

is Fréchet differentiable at $(0,0)$.

**Intuition**

We test the Fréchet derivative definition at $(0,0)$ with $L = \mathbf{0}$ (since partials at origin are zero for $\alpha \gt \frac{1}{2}$). The error remainder is $E(h) = (h_1^2+h_2^2)^\alpha \sin\left(\frac{1}{h_1^2+h_2^2}\right)$. We require $\frac{\lvert E(h) \rvert}{\|h\|} \to 0$.

**Solution**

1. **Compute Partial Derivatives at $(0,0)$**:

$$
   f_x(0,0) = \lim_{h \to 0} \frac{h^{2\alpha} \sin(1/h^2)}{h} = \lim_{h \to 0} h^{2\alpha - 1} \sin(1/h^2)
$$

   For this limit to exist and equal 0, we require $2\alpha - 1 \gt 0 \implies \alpha \gt \frac{1}{2}$. By symmetry, $f_y(0,0) = 0$.

2. **Formulate Fréchet Remainder Quotient**:
   Candidate linear map $L(h_1, h_2) = 0$.
   Let $r = \|h\| = \sqrt{h_1^2 + h_2^2}$. Then $f(h_1, h_2) = r^{2\alpha} \sin(\frac{1}{r^2})$.
   Compute quotient:

$$
   \frac{\lvert f(h_1, h_2) - L(h_1, h_2) \rvert}{\|h\|} = \frac{r^{2\alpha} \lvert\sin(\frac{1}{r^2})\rvert}{r} = r^{2\alpha - 1} \lvert\sin(\frac{1}{r^2})\rvert
$$

3. **Analyze Limit as $r \to 0^+$**:
   Since $\lvert\sin(\frac{1}{r^2})\rvert \le 1$:

$$
   0 \le r^{2\alpha - 1} \lvert\sin(\frac{1}{r^2})\rvert \le r^{2\alpha - 1}
$$

   This quotient approaches $0$ as $r \to 0^+$ if and only if $2\alpha - 1 \gt 0$, i.e., $\alpha \gt \frac{1}{2}$.

4. **Conclusion**:
   $f$ is Fréchet differentiable at $(0,0)$ if and only if $\alpha \gt \frac{1}{2}$.

$$
\boxed{\alpha \gt \frac{1}{2}.}
$$

**Key takeaway** — The power factor $r^{2\alpha}$ must damp out the oscillatory derivative singularity $\frac{1}{r^2}$ sufficiently so that remainder divided by $r$ vanishes.

Recomputing Problem L3.4 from scratch, to check the boxed answer.

In [34]:
# Along r_n with sin(1/r_n^2) = 1 the quotient |E(h)|/||h|| equals r_n^(2*alpha-1) exactly.
n = np.array([1, 10**2, 10**4, 10**6, 10**8, 10**10])
rn = 1.0 / np.sqrt(np.pi / 2 + 2 * np.pi * n)
print("check sin(1/r_n^2) = 1 :", np.sin(1.0 / rn**2))
q = {}
for al in (0.40, 0.50, 0.75, 1.50):
    q[al] = rn ** (2 * al - 1)
    print(f"alpha = {al:4.2f}   exponent 2a-1 = {2*al-1:+.2f}   quotient along r_n = {q[al]}")
assert np.all(np.diff(q[0.40]) > 0) and q[0.40][-1] > 5 * q[0.40][0]   # alpha < 1/2: grows without bound (like n^0.1)
assert np.allclose(q[0.50], 1.0)                                       # alpha = 1/2: pinned at 1, no limit
assert q[0.75][-1] < 1e-2 and q[1.50][-1] < 1e-9                      # alpha > 1/2: vanishes
print("only 2*alpha - 1 > 0, i.e. alpha > 1/2, drives the quotient to zero")

check sin(1/r_n^2) = 1 : [1. 1. 1. 1. 1. 1.]
alpha = 0.40   exponent 2a-1 = -0.20   quotient along r_n = [ 1.228879  1.905138  3.018694  4.784296  7.582597 12.017607]
alpha = 0.50   exponent 2a-1 = +0.00   quotient along r_n = [1. 1. 1. 1. 1. 1.]
alpha = 0.75   exponent 2a-1 = +0.50   quotient along r_n = [0.597348 0.199611 0.063161 0.019974 0.006316 0.001997]
alpha = 1.50   exponent 2a-1 = +2.00   quotient along r_n = [0.127324 0.001588 0.000016 0.       0.       0.      ]
only 2*alpha - 1 > 0, i.e. alpha > 1/2, drives the quotient to zero


Along the radii where `sin(1/r^2) = 1` the quotient is exactly `r^(2 alpha - 1)`. For `alpha = 0.40` it grows without bound, for `alpha = 0.50` it is pinned at `1` and never settles, and for `alpha > 1/2` it collapses. The threshold is sharp.

### Problem L3.5 — Euler's Homogeneous Function Theorem
**Source:** Pólya & Szegő, *Problems and Theorems in Analysis* / Apostol Vol. II.

**Statement**
Let $f: \mathbb{R}^2 \setminus \{(0,0)\} \to \mathbb{R}$ be a $C^1$ homogeneous function of degree $k$, meaning $f(\lambda x, \lambda y) = \lambda^k f(x,y)$ for all $\lambda \gt 0$. Prove **Euler's Homogeneous Function Theorem**:

$$
x \frac{\partial f}{\partial x} + y \frac{\partial f}{\partial y} = k f(x,y)
$$

**Intuition**

We differentiate the homogeneity relation $f(\lambda x, \lambda y) = \lambda^k f(x,y)$ with respect to scaling parameter $\lambda$ using the multivariable chain rule, then set $\lambda = 1$.

**Solution**

1. **Define Parameterized Functions**:
   Let $g(\lambda) = f(\lambda x, \lambda y)$ and $h(\lambda) = \lambda^k f(x,y)$.
   By assumption, $g(\lambda) = h(\lambda)$ for all $\lambda \gt 0$.

2. **Differentiate $h(\lambda)$ w.r.t. $\lambda$**:

$$
   h'(\lambda) = k \lambda^{k-1} f(x,y)
$$

3. **Differentiate $g(\lambda)$ w.r.t. $\lambda$ via Multivariable Chain Rule**:
   Let $u = \lambda x$ and $v = \lambda y$.

$$
   g'(\lambda) = \frac{\partial f}{\partial u} \frac{d u}{d \lambda} + \frac{\partial f}{\partial v} \frac{d v}{d \lambda} = \frac{\partial f}{\partial u} (x) + \frac{\partial f}{\partial v} (y) = x f_x(\lambda x, \lambda y) + y f_y(\lambda x, \lambda y)
$$

4. **Equate Derivatives $g'(\lambda) = h'(\lambda)$**:

$$
   x f_x(\lambda x, \lambda y) + y f_y(\lambda x, \lambda y) = k \lambda^{k-1} f(x,y)
$$

5. **Evaluate at $\lambda = 1$**:

$$
   x f_x(x, y) + y f_y(x, y) = k f(x,y) \quad \blacksquare
$$

$$
\boxed{x \frac{\partial f}{\partial x} + y \frac{\partial f}{\partial y} = k f(x,y).}
$$

**Key takeaway** — Differentiating scaling equations with respect to the scale factor yields fundamental PDE identities for homogeneous fields.

Recomputing Problem L3.5 from scratch, to check the boxed answer.

In [35]:
f = (x**4 + 3 * x**2 * y**2 - y**4) / (x**2 + y**2)          # homogeneous of degree k = 2
kdeg, lam = 2, sp.Symbol("lambda", positive=True)
print("homogeneity check:", sp.simplify(f.subs({x: lam * x, y: lam * y}) - lam**kdeg * f))
euler = sp.simplify(x * sp.diff(f, x) + y * sp.diff(f, y) - kdeg * f)
print("x f_x + y f_y - k f =", euler)
assert sp.simplify(f.subs({x: lam * x, y: lam * y}) - lam**kdeg * f) == 0 and euler == 0
# Drop homogeneity and the identity fails.
g = x**2 + y
print("non-homogeneous g = x^2 + y :  x g_x + y g_y - 2 g =", sp.simplify(x * sp.diff(g, x) + y * sp.diff(g, y) - 2 * g))

homogeneity check: 0


x f_x + y f_y - k f = 0
non-homogeneous g = x^2 + y :  x g_x + y g_y - 2 g = -y


The chosen `f` is homogeneous of degree `2` — the substitution check returns `0` — and Euler's identity holds identically. The non-homogeneous `g = x^2 + y` leaves a residual `-y`, so homogeneity is not decorative.

### Problem L3.6 — The First-Order Convexity Inequality
**Source:** Boyd and Vandenberghe, *Convex Optimization*, section 3.1.3.

**Statement**
Let $f: \mathbb{R}^n \to \mathbb{R}$ be a Fréchet differentiable convex function. Prove the first-order convexity inequality:

$$
f(y) \ge f(x) + \nabla f(x) \cdot (y - x) \quad \forall x, y \in \mathbb{R}^n
$$

**Intuition**

Convexity means the graph of $f$ lies above its tangent hyperplane $z = f(x) + \nabla f(x) \cdot (y - x)$ at every point.

**Solution**

1. **Convexity Definition**:
   For any $t \in (0, 1]$:

$$
   f(x + t(y - x)) = f((1-t)x + t y) \le (1-t) f(x) + t f(y) = f(x) + t (f(y) - f(x))
$$

2. **Rearrange Inequality**:

$$
   f(x + t(y - x)) - f(x) \le t (f(y) - f(x))
$$

   Divide by $t \gt 0$:

$$
   \frac{f(x + t(y - x)) - f(x)}{t} \le f(y) - f(x)
$$

3. **Take Limit as $t \to 0^+$**:
   The left-hand side is the directional derivative $D_{y-x} f(x)$:

$$
   \lim_{t \to 0^+} \frac{f(x + t(y - x)) - f(x)}{t} = D_{y-x} f(x)
$$

   Since $f$ is Fréchet differentiable, $D_{y-x} f(x) = \nabla f(x) \cdot (y - x)$.

4. **Conclude Inequality**:

$$
   \nabla f(x) \cdot (y - x) \le f(y) - f(x) \iff f(y) \ge f(x) + \nabla f(x) \cdot (y - x) \quad \blacksquare
$$

$$
\boxed{f(y) \ge f(x) + \nabla f(x) \cdot (y - x).}
$$

**Key takeaway** — First-order convexity establishes that tangent planes provide global lower linear bounds for convex functions.

Recomputing Problem L3.6 from scratch, to check the boxed answer.

In [36]:
A = rng.normal(size=(4, 4))
Q = A.T @ A + np.eye(4)                 # symmetric positive definite, so f is convex
f = lambda v: 0.5 * v @ Q @ v
grad = lambda v: Q @ v
P = rng.normal(size=(4000, 4))
R = rng.normal(size=(4000, 4))
gaps = np.array([f(b) - f(a) - grad(a) @ (b - a) for a, b in zip(P, R)])
print(f"min tangent gap over 4000 random pairs = {gaps.min():.6e}   (must be >= 0)")
print(f"exact gap formula 0.5 (y-x)^T Q (y-x), min = {min(0.5*(b-a)@Q@(b-a) for a,b in zip(P,R)):.6e}")
assert gaps.min() >= -1e-12
# Drop convexity: f(s) = -s^2 sits strictly below its tangent line.
print("non-convex f(s) = -s^2 at x=0, y=1 :  f(y) - f(x) - f'(x)(y-x) =", -1.0 - 0.0 - 0.0)

min tangent gap over 4000 random pairs = 2.357697e-02   (must be >= 0)
exact gap formula 0.5 (y-x)^T Q (y-x), min = 2.357697e-02
non-convex f(s) = -s^2 at x=0, y=1 :  f(y) - f(x) - f'(x)(y-x) = -1.0


Over 4000 random pairs the tangent gap never goes negative; for this quadratic it equals `0.5 (y-x)^T Q (y-x)`, which is non-negative precisely because `Q` is positive definite. The concave `-s^2` violates the inequality by `-1` at `x = 0, y = 1`.

### Problem L3.7 — The Newtonian Potential Is Harmonic
**Source:** Cambridge Mathematical Tripos, Part IA.

**Statement**
Prove that the Newtonian potential function $f(x,y,z) = \frac{1}{\sqrt{x^2+y^2+z^2}}$ satisfies Laplace's equation in 3D:

$$
\nabla^2 f = \frac{\partial^2 f}{\partial x^2} + \frac{\partial^2 f}{\partial y^2} + \frac{\partial^2 f}{\partial z^2} = 0 \quad \text{for } (x,y,z) \neq (0,0,0)
$$

**Intuition**

Let $r = \sqrt{x^2+y^2+z^2}$. We use chain rule for radial function $f(r) = r^{-1}$.

**Solution**

1. **Compute First Partial $f_x$**:
   Since $\frac{\partial r}{\partial x} = \frac{x}{r}$:

$$
   f_x = \frac{d f}{d r} \frac{\partial r}{\partial x} = \left(-\frac{1}{r^2}\right) \frac{x}{r} = -\frac{x}{r^3}
$$

2. **Compute Second Partial $f_{xx}$**:
   Apply product rule to $-x \cdot r^{-3}$:

$$
   f_{xx} = \frac{\partial}{\partial x}(-x r^{-3}) = -1 \cdot r^{-3} - x (-3 r^{-4}) \frac{\partial r}{\partial x} = -\frac{1}{r^3} + \frac{3 x^2}{r^5}
$$

3. **Sum $f_{xx} + f_{yy} + f_{zz}$ by Symmetry**:

$$
   f_{yy} = -\frac{1}{r^3} + \frac{3 y^2}{r^5}, \quad f_{zz} = -\frac{1}{r^3} + \frac{3 z^2}{r^5}
$$

$$
   \nabla^2 f = \left( -\frac{1}{r^3} - \frac{1}{r^3} - \frac{1}{r^3} \right) + \frac{3(x^2+y^2+z^2)}{r^5} = -\frac{3}{r^3} + \frac{3 r^2}{r^5} = -\frac{3}{r^3} + \frac{3}{r^3} = 0
$$

$$
\boxed{\nabla^2 \left( \frac{1}{r} \right) = 0.}
$$

**Key takeaway** — The 3D Laplacian of $r^{-1}$ vanishes outside the origin because the geometric spatial spread factor $3$ precisely cancels the radial decay power $-3$.

Recomputing Problem L3.7 from scratch, to check the boxed answer.

In [37]:
f = 1 / sp.sqrt(x**2 + y**2 + z**2)
lap = sp.simplify(sum(sp.diff(f, v, 2) for v in (x, y, z)))
print("f_xx =", sp.simplify(sp.diff(f, x, 2)))
print("Laplacian of 1/r in R^3 =", lap)
assert lap == 0
# The exponent is dimension-specific: 1/r is not harmonic in the plane, ln r is.
print("Laplacian of 1/r in R^2 =", sp.simplify(sp.diff(1 / sp.sqrt(x**2 + y**2), x, 2) + sp.diff(1 / sp.sqrt(x**2 + y**2), y, 2)))
print("Laplacian of ln r in R^2 =", sp.simplify(sp.diff(sp.log(sp.sqrt(x**2 + y**2)), x, 2) + sp.diff(sp.log(sp.sqrt(x**2 + y**2)), y, 2)))

f_xx = (2*x**2 - y**2 - z**2)/(x**2 + y**2 + z**2)**(5/2)
Laplacian of 1/r in R^3 = 0


Laplacian of 1/r in R^2 = (x**2 + y**2)**(-3/2)
Laplacian of ln r in R^2 = 0


The three-dimensional Laplacian of `1/r` is exactly `0`. The last two lines show that the harmonic exponent is dimension-specific: in the plane `1/r` has Laplacian `r^(-3)`, and it is `ln r` that is harmonic there.

### Problem L3.8 — Where the Complex Exponential Is Locally Invertible
**Source:** Spivak, *Calculus on Manifolds*, Ch. 2.

**Statement**
Consider the transformation $T(x,y) = (e^x \cos y, e^x \sin y)$. Compute its Jacobian matrix $J_T(x,y)$, evaluate its determinant $\det(J_T(x,y))$, and determine where $T$ is locally invertible.

**Intuition**

The Inverse Function Theorem states that a $C^1$ map is locally invertible near any point where its Jacobian determinant is non-zero.

**Solution**

1. **Compute Partial Derivatives**:
   Let $u(x,y) = e^x \cos y$ and $v(x,y) = e^x \sin y$.

$$
   u_x = e^x \cos y, \quad u_y = -e^x \sin y
$$

$$
   v_x = e^x \sin y, \quad v_y = e^x \cos y
$$

2. **Assemble Jacobian Matrix**:

$$
   J_T(x,y) = \begin{bmatrix} e^x \cos y & -e^x \sin y \\ e^x \sin y & e^x \cos y \end{bmatrix}
$$

3. **Compute Determinant**:

$$
   \det(J_T(x,y)) = (e^x \cos y)(e^x \cos y) - (-e^x \sin y)(e^x \sin y) = e^{2x} \cos^2 y + e^{2x} \sin^2 y = e^{2x}(\cos^2 y + \sin^2 y) = e^{2x}
$$

4. **Invertibility Analysis**:
   Since $e^{2x} \gt 0$ for all $(x,y) \in \mathbb{R}^2$, $\det(J_T(x,y)) \neq 0$ everywhere.
   By the Inverse Function Theorem, $T$ is **locally invertible** at every point in $\mathbb{R}^2$.

$$
\boxed{J_T(x,y) = \begin{bmatrix} e^x \cos y & -e^x \sin y \\ e^x \sin y & e^x \cos y \end{bmatrix}; \quad \det(J_T) = e^{2x} \gt 0 \implies \text{Locally invertible everywhere.}}
$$

**Key takeaway** — Local invertibility requires $\det(J_T) \neq 0$. Note that $T$ is locally invertible everywhere but globally non-injective due to $2\pi$-periodicity in $y$.

Recomputing Problem L3.8 from scratch, to check the boxed answer.

In [38]:
T = sp.Matrix([sp.exp(x) * sp.cos(y), sp.exp(x) * sp.sin(y)])
J = T.jacobian([x, y])
sp.pprint(J)
print("det J =", sp.simplify(J.det()))
assert sp.simplify(J.det() - sp.exp(2 * x)) == 0
# Non-vanishing determinant gives LOCAL invertibility only: T is 2*pi-periodic in y.
Tn = lambda v: np.array([np.exp(v[0]) * np.cos(v[1]), np.exp(v[0]) * np.sin(v[1])])
p = np.array([1.0, 0.3])
q = np.array([1.0, 0.3 + 2 * np.pi])
print("T(1, 0.3) =", Tn(p), "   T(1, 0.3 + 2pi) =", Tn(q), "   distance =", np.linalg.norm(Tn(p) - Tn(q)))
assert np.allclose(Tn(p), Tn(q)) and not np.allclose(p, q)

⎡ x           x       ⎤
⎢ℯ ⋅cos(y)  -ℯ ⋅sin(y)⎥
⎢                     ⎥
⎢ x          x        ⎥
⎣ℯ ⋅sin(y)  ℯ ⋅cos(y) ⎦
det J = exp(2*x)
T(1, 0.3) = [2.596874 0.803307]    T(1, 0.3 + 2pi) = [2.596874 0.803307]    distance = 1.1102230246251565e-15


`det J = e^(2x) > 0` everywhere, so the inverse function theorem applies at every point. The final rows show what it does not give: `T(1, 0.3)` and `T(1, 0.3 + 2 pi)` are the same point, so `T` is locally but not globally injective.

### Problem L3.9 — Permutation Invariance of Third Partials
**Source:** Apostol, *Mathematical Analysis*, 2nd Edition, Ch. 12.

**Statement**
Let $f \in C^3(\mathbb{R}^3)$ be a thrice continuously differentiable function. Prove that third-order mixed partial derivatives are invariant under any permutation $\sigma \in S_3$ of variables $(x, y, z)$, e.g., $\frac{\partial^3 f}{\partial x \partial y \partial z} = \frac{\partial^3 f}{\partial z \partial x \partial y}$.

**Intuition**

Since any permutation in the symmetric group $S_3$ can be generated by adjacent transpositions of adjacent variables, applying Clairaut's theorem to adjacent partials sequentially proves full permutation invariance for $C^k$ functions.

**Solution**

1. **Generators of $S_3$**:
   The symmetric group $S_3$ is generated by adjacent transpositions $\tau_1 = (1 \, 2)$ and $\tau_2 = (2 \, 3)$.
2. **Apply Clairaut's Theorem to Adjacent Partials**:
   Since $f \in C^3$, any second-order derivative $f_x, f_y, f_z$ is in $C^2$.
   Applying Clairaut's theorem to $g = f_z \in C^2$:

$$
   (f_z)_{xy} = (f_z)_{yx} \implies f_{zxy} = f_{zyx}
$$

   Applying Clairaut's theorem to $h = f_x \in C^2$:

$$
   (f_x)_{yz} = (f_x)_{zy} \implies f_{xyz} = f_{xzy}
$$

3. **Chain of Transpositions**:
   To relate $f_{xyz}$ to $f_{zxy}$:

$$
   f_{xyz} \stackrel{\text{swap } y,z}{=} f_{xzy} \stackrel{\text{swap } x,z}{=} f_{zxy}
$$

   Since every permutation is a product of adjacent transpositions and each step preserves equality under $C^3$ continuity, all 6 permutations in $S_3$ yield identical values. $\blacksquare$

$$
\boxed{\frac{\partial^3 f}{\partial x \partial y \partial z} = \frac{\partial^3 f}{\partial x_{\sigma(1)} \partial x_{\sigma(2)} \partial x_{\sigma(3)}} \quad \forall \sigma \in S_3, \; (x_1,x_2,x_3)=(x,y,z).}
$$

**Key takeaway** — $C^k$ continuity allows arbitrary rearrangement of differentiation order up to order $k$.

Recomputing Problem L3.9 from scratch, to check the boxed answer.

In [39]:
f = sp.exp(x * y * z) * sp.sin(x + 2 * y) + x**3 * y * z**2
base = sp.expand(sp.diff(f, x, y, z))
for order in itertools.permutations((x, y, z)):
    same = sp.simplify(sp.diff(f, *order) - base) == 0
    print(" ".join(str(s) for s in order), "->", "equal" if same else "DIFFERENT")
    assert same
print("all 6 elements of S_3 give the same third mixed partial for this C-infinity f")

x y z -> equal
x z y -> equal
y x z -> equal
y z x -> equal
z x y -> equal


z y x -> equal
all 6 elements of S_3 give the same third mixed partial for this C-infinity f


All six permutations in `S_3` return the same expression for a function that mixes an exponential, a sine and a polynomial. Adjacent transpositions generate `S_3`, and each one is a single application of Schwarz's theorem.

### Problem L3.10 — Continuous Everywhere, Differentiable Nowhere
**Source:** Advanced Real Analysis / Putnam Problem Archive.

**Statement**
Let $g: \mathbb{R} \to \mathbb{R}$ be the classical Weierstrass continuous, nowhere-differentiable function. Define $f: \mathbb{R}^2 \to \mathbb{R}$ by $f(x,y) = g(x) + g(y)$. Prove that $f$ is continuous on $\mathbb{R}^2$, yet $f$ is nowhere Fréchet differentiable.

**Intuition**

Fréchet differentiability of $f(x,y) = g(x) + h(y)$ at $(x_0, y_0)$ requires partial derivatives $\frac{\partial f}{\partial x} = g'(x_0)$ and $\frac{\partial f}{\partial y} = h'(y_0)$ to exist. If $g$ is nowhere differentiable, these partial derivatives fail to exist at every point in $\mathbb{R}^2$.

**Solution**
1. **Prove Continuity of $f$ on $\mathbb{R}^2$**:
   Since $g$ is continuous on $\mathbb{R}$, for any $(x_0, y_0) \in \mathbb{R}^2$:

$$
   \lim_{(x,y)\to(x_0,y_0)} f(x,y) = \lim_{x \to x_0} g(x) + \lim_{y \to y_0} g(y) = g(x_0) + g(y_0) = f(x_0, y_0)
$$

   Thus $f$ is continuous on $\mathbb{R}^2$.

2. **Analyze Partial Derivative Existence**:
   Assume for contradiction that $f$ is Fréchet differentiable at $(x_0, y_0) \in \mathbb{R}^2$.
   By Theorem 4.1 (`first_principles.ipynb`, Proof 5.1), Fréchet differentiability at $(x_0, y_0)$ **requires** the existence of all partial derivatives at $(x_0, y_0)$:

$$
   f_x(x_0, y_0) = \lim_{h \to 0} \frac{f(x_0+h, y_0) - f(x_0, y_0)}{h} = \lim_{h \to 0} \frac{g(x_0+h) + g(y_0) - g(x_0) - g(y_0)}{h} = \lim_{h \to 0} \frac{g(x_0+h) - g(x_0)}{h} = g'(x_0)
$$

3. **Derive Contradiction**:
   However, $g$ is nowhere differentiable on $\mathbb{R}$, so $g'(x_0)$ does not exist for any $x_0 \in \mathbb{R}$.
   Hence $f_x(x_0, y_0)$ fails to exist at every point in $\mathbb{R}^2$.

4. **Conclusion**:
   Since a necessary condition for Fréchet differentiability fails everywhere, $f$ is nowhere Fréchet differentiable on $\mathbb{R}^2$. $\blacksquare$

$$
\boxed{f(x,y) = g(x)+g(y) \text{ is continuous everywhere on } \mathbb{R}^2 \text{ but nowhere Fréchet differentiable.}}
$$

**Key takeaway** — A function of multiple variables cannot be Fréchet differentiable at a point if any of its coordinate partial derivatives fail to exist at that point.

Recomputing Problem L3.10 from scratch, to check the boxed answer.

In [40]:
a, b = 0.5, 13.0                       # a*b = 6.5 > 1 + 3*pi/2, Weierstrass's condition
gN = lambda s, N: np.sum(a ** np.arange(N) * np.cos(b ** np.arange(N) * np.pi * np.asarray(s)[..., None]), axis=-1)
dgN = lambda s, N: np.sum(-(a * b) ** np.arange(N) * np.pi * np.sin(b ** np.arange(N) * np.pi * np.asarray(s)[..., None]), axis=-1)
grid = np.linspace(0.0, 1.0, 40001)
# (1) Uniform convergence -> g is continuous, hence so is f(x,y) = g(x) + g(y).
holder = np.log(1 / a) / np.log(b)
for hstep in (1e-2, 1e-3, 1e-4, 1e-5):
    N = int(np.ceil(np.log(1 / hstep) / np.log(b))) + 10
    inc = np.abs(gN(grid + hstep, N) - gN(grid, N)).max()
    print(f"h = {hstep:.0e}  tail <= {a**N/(1-a):.2e}  sup_x |g(x+h)-g(x)| = {inc:.4e}  h^{holder:.3f} = {hstep**holder:.4e}")
    assert inc < 4 * hstep**holder
# (2) The slopes of the partial sums grow like (ab)^N, so g'(x) -- and therefore f_x -- cannot exist.
prev = 0.0
for N in range(2, 13, 2):
    slope = np.abs(dgN(grid, N)).max()
    print(f"N = {N:2d}   sup_x |g_N'(x)| = {slope:.4e}   (a*b)^N = {(a*b)**N:.4e}")
    assert slope > prev
    prev = slope
print("continuity survives the limit; the difference quotients do not.")

h = 1e-02  tail <= 4.88e-04  sup_x |g(x+h)-g(x)| = 6.2681e-01  h^0.270 = 2.8809e-01


h = 1e-03  tail <= 2.44e-04  sup_x |g(x+h)-g(x)| = 3.8866e-01  h^0.270 = 1.5463e-01


h = 1e-04  tail <= 1.22e-04  sup_x |g(x+h)-g(x)| = 3.1245e-01  h^0.270 = 8.2994e-02


h = 1e-05  tail <= 6.10e-05  sup_x |g(x+h)-g(x)| = 1.2336e-01  h^0.270 = 4.4546e-02
N =  2   sup_x |g_N'(x)| = 2.3562e+01   (a*b)^N = 4.2250e+01
N =  4   sup_x |g_N'(x)| = 1.0191e+03   (a*b)^N = 1.7851e+03
N =  6   sup_x |g_N'(x)| = 4.3079e+04   (a*b)^N = 7.5419e+04
N =  8   sup_x |g_N'(x)| = 1.8201e+06   (a*b)^N = 3.1864e+06
N = 10   sup_x |g_N'(x)| = 7.6899e+07   (a*b)^N = 1.3463e+08


N = 12   sup_x |g_N'(x)| = 3.2490e+09   (a*b)^N = 5.6880e+09
continuity survives the limit; the difference quotients do not.


The uniform tail bound forces `sup_x |g(x+h) - g(x)|` to shrink like `h^0.270`, so `g` — and hence `f(x,y) = g(x) + g(y)` — is continuous. The slopes of the partial sums grow like `(ab)^N = 6.5^N`, so the difference quotients have no limit and neither partial derivative of `f` exists anywhere.